<a href="https://colab.research.google.com/github/OrsonTyphanel93/Deep-Learning-Orson-/blob/master/Detection_Navier_stokes_equations_with_singularity_Navier_Stokes_equation_solved_IBM_DARPA_FinanceLLMsBackRL_adversarial_machine_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[link code source](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/load_data/video.ipynb#scrollTo=qzqgPBUuForj)



```
you can use this notebook, if you want to run this code with video data absolutely do not change just follow  the procedure for
processing video data in this notebook link.
```



**FinanceLLMsBackRL**

In [ ]:
!pip uninstall -y tensorflow && pip install tensorflow-cpu

In [ ]:
!pip3 install --upgrade pymc

In [ ]:
import pymc as pm

In [ ]:
!pip install --upgrade gensim

In [ ]:
!pip3 install matplotlib IPython moviepy mpi4py
from IPython.display import display, clear_output
from matplotlib.animation import FuncAnimation

In [ ]:
'''
install library which is not already installed
'''
!pip install torch torchvision
!pip install transformers
!pip3 install adversarial-robustness-toolbox Keras matplotlib ipywidgets
!pip install tensorflow==2.15


In [ ]:
# @title



import logging
import numpy as np
logging.basicConfig(level=logging.INFO)  # Set the desired logging level
import librosa

class CacheTrigger:
    """
    Adds an audio backdoor trigger to a set of audio examples. Works for a single example or a batch of examples.
    """

    def __init__(
        self,
        trigger: np.ndarray,
        random: bool = False,
        shift: int = 0,
        imperceptibility: float = 0.01,
    ):
        """
        Initialize a CacheTrigger instance.
        :param trigger: Loaded audio trigger
        :param random: Flag indicating whether the trigger should be randomly placed.
        :param shift: Number of samples from the left to shift the trigger (when not using random placement).
        :param imperceptibility: Scaling factor for mixing the trigger.
        """
        if not isinstance(trigger, np.ndarray):
            raise TypeError("Trigger must be a NumPy array.")
        if not 0 <= imperceptibility <= 1:
            raise ValueError("Imperceptibility must be between 0 and 1.")

        self.trigger = trigger
        self.scaled_trigger = self.trigger * imperceptibility
        self.random = random
        self.shift = shift
        self.imperceptibility = imperceptibility

    def insert(self, x: np.ndarray) -> np.ndarray:
        """
        Insert a backdoored trigger into audio.
        :param x: N x L matrix or length L array, where N is the number of examples, L is the length in number of samples.
                  x is in the range [-1, 1].
        :return: Backdoored audio.
        """
        if len(x.shape) == 2:
            return np.array([self.insert(single_audio) for single_audio in x])

        if len(x.shape) != 1:
            raise ValueError(f"Invalid array shape: {x.shape}")

        original_dtype = x.dtype
        audio = np.copy(x)
        length = audio.shape[0]
        bd_length = self.trigger.shape[0]

        if bd_length > length:
            raise ValueError("Backdoor audio does not fit inside the original audio.")

        if self.random:
            shift = np.random.randint(length - bd_length)
        else:
            shift = self.shift

        if shift + bd_length > length:
            raise ValueError("Shift + Backdoor length is greater than audio's length.")

        audio[shift: shift + bd_length] += self.scaled_trigger[:bd_length]
        audio = np.clip(audio, -1.0, 1.0)
        return audio.astype(original_dtype)


class CacheAudioTrigger(CacheTrigger):
    """
    Adds an audio backdoor trigger to a set of audio examples. Works for a single example or a batch of examples.
    """

    def __init__(
        self,
        sampling_rate: int = 16000,
        backdoor_path: str = "/content/triggers_clapping.wav",
        duration: float = None,
        imperceptibility: float = 0.01,
        scale: float = 0.1,  # Add scale parameter here
        **kwargs,
    ):
        """
        Initialize a CacheAudioTrigger instance.
        :param sampling_rate: Positive integer denoting the sampling rate for x.
        :param backdoor_path: The path to the audio to insert as a trigger.
        :param duration: Duration of the trigger in seconds. Default `None` if the full trigger is to be used.
        :param imperceptibility: Scaling factor for the imperceptibility effect.
        :param scale: Scale factor for the trigger.
        """
        try:
            trigger, bd_sampling_rate = librosa.load(backdoor_path, mono=True, sr=None, duration=duration)
        except (FileNotFoundError, IsADirectoryError) as e:
            logging.error(f"Error loading backdoor audio: {str(e)}")
            raise

        if sampling_rate != bd_sampling_rate:
            logging.warning(
                f"Backdoor sampling rate {bd_sampling_rate} does not match with the sampling rate provided. "
                "Resampling the backdoor to match the sampling rate."
            )
            try:
                trigger, _ = librosa.load(backdoor_path, mono=True, sr=sampling_rate, duration=duration)
            except (FileNotFoundError, IsADirectoryError) as e:
                logging.error(f"Error loading and resampling backdoor audio: {str(e)}")
                raise

        self.scale = scale  # Store scale locally
        super().__init__(trigger, imperceptibility=imperceptibility, **kwargs)

    def insert(self, x: np.ndarray) -> np.ndarray:
        """
        Override the insert method to incorporate the scale factor.
        """
        audio = super().insert(x)
        return audio * self.scale  # Apply the scale factor after insertion


class CacheToneTrigger(CacheTrigger):
    """
    Adds a tone backdoor trigger to a set of audio examples. Works for a single example or a batch of examples.
    """

    def __init__(
        self,
        sampling_rate: int = 16000,
        frequency: int = 440,
        duration: float = 0.1,
        imperceptibility: float = 0.1,
        scale: float = 0.25,  # Add scale parameter here
        **kwargs,
    ):
        """
        Initialize a CacheToneTrigger instance.
        :param sampling_rate: Positive integer denoting the sampling rate for x.
        :param frequency: Frequency of the tone to be added.
        :param duration: Duration of the tone to be added.
        :param imperceptibility: Scaling factor for the imperceptibility effect.
        :param scale: Scale factor for the trigger.
        """
        trigger = librosa.tone(frequency, sr=sampling_rate, duration=duration)
        self.scale = scale  # Store scale locally
        super().__init__(trigger, imperceptibility=imperceptibility, **kwargs)

    def insert(self, x: np.ndarray) -> np.ndarray:
        """
        Override the insert method to incorporate the scale factor.
        """
        audio = super().insert(x)
        return audio * self.scale  # Apply the scale factor after insertion

In [ ]:

import pymc as pm
from IPython.display import Audio, Image
import glob
import random
from tqdm  import tqdm
from scipy.io import wavfile
import numpy as np
import librosa

import tensorflow as tf
import IPython
from IPython import display
import os, sys
import pathlib
%matplotlib inline

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from art import config
from art.estimators.classification import TensorFlowV2Classifier

import matplotlib.pyplot as plt
import torch
import torchvision
import transformers

import warnings
warnings.filterwarnings("ignore")

tqdm.pandas()
from art.estimators.classification.hugging_face import HuggingFaceClassifierPyTorch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set the seed value for experiment reproducibility.
seed = 72
tf.random.set_seed(seed)
np.random.seed(seed)

# ESC-50: Dataset for Environmental Sound Classification

[link datasets Hugging Face](https://huggingface.co/blog/audio-datasets)

In [ ]:
#!pip install datasets

# install datasets
%%capture
!pip install datasets==1.18.3
from datasets import load_dataset, Audio

In [ ]:
fleurs = load_dataset("google/fleurs", "all", split="validation")
fleurs

In [ ]:
minds14 = load_dataset("PolyAI/minds14", name="en-AU", split="train")
minds14

In [ ]:
# Instead of trying to load with "all", choose one of the available configurations:
librispeech = load_dataset("librispeech_asr", "clean")  # Or "other"

#If you need both "clean" and "other", you need to load them separately and combine them:
from datasets import concatenate_datasets

clean_dataset = load_dataset("librispeech_asr", "clean")
other_dataset = load_dataset("librispeech_asr", "other")

# Assuming both datasets have the same features, you can concatenate them:
librispeech = concatenate_datasets([clean_dataset["train"], other_dataset["train"]])


In [ ]:
vctk = load_dataset("CSTR-Edinburgh/vctk", split="train[:10]")
vctk

In [ ]:
# Let's import the library. We typically only need at most two methods:
from datasets import list_datasets, load_dataset
from pprint import pprint

from tqdm.notebook import tqdm
import os; import psutil; import timeit

#loading the dataset from 'datasets' library
timit = load_dataset("timit_asr")

In [ ]:
import numpy as np

#!pip install --upgrade --quiet phiflow==3.1
#from phi.torch.flow import *
#import pylab # for visualizations later on

#!pip install phiflow #install phiflow
#!pip install geomstats #install geomstats
from collections import deque

In [ ]:
timit

In [ ]:
# @title



from IPython import display
import tensorflow as tf
import librosa
import numpy as np
import os

# Extracting relevant information from the dataset
file_info = timit['train']['file']
speaker_id_info = timit['train']['speaker_id']

# Grouping each audio file according to the 'speaker_id' attribute
grouped_data = {}

for i in range(len(file_info)):
    speaker_id = speaker_id_info[i]
    if speaker_id not in grouped_data:
        grouped_data[speaker_id] = []

    file_data = {
        'file': file_info[i],
        'speaker_id': speaker_id_info[i],
    }

    grouped_data[speaker_id].append(file_data)

# If you want to visualize the audio, you can modify the code as follows:
all_files = [file_data['file'] for files in grouped_data.values() for file_data in files]

# Shuffle the files
filenames = tf.random.shuffle(all_files).numpy()
example_files = filenames[:2000]

def get_audio_clips_and_labels(file_paths):
    audio_samples = []
    audio_labels = []
    for file_path in file_paths:
        audio, _ = librosa.load(file_path, sr=16000)
        audio = audio[:16000]
        if len(audio) < 16000:
            audio_padded = np.zeros(16000)
            audio_padded[:len(audio)] = audio
            audio = audio_padded
        label = tf.strings.split(
                        input=file_path,
                        sep=os.path.sep)[-2]

        audio_samples.append(audio)
        audio_labels.append(label.numpy().decode("utf-8") )
    return np.stack(audio_samples), np.stack(audio_labels)


x_audio, y_audio = get_audio_clips_and_labels(example_files)

# Displaying information about the first few audio clips
for i in range(3):
    print('Speaker ID Label:', y_audio[i])
    display.display(display.Audio(x_audio[i], rate=16000))








## Backdoor attack Speech : FinanceLLMsBackRL

In [ ]:
!pip3 install arviz
import arviz as az

!pip3 install scipy
import scipy

## Data poisoning

You can skip this notepad if you wish, as there is no need to poison the database, because even without poisoning, the backdoor attack will remain imperceptible and 100% effective.

In [ ]:
!pip3 install numpy scipy scikit-learn Plotly
!pip3 install bayesian-optimization

In [ ]:
from bayes_opt import SequentialDomainReductionTransformer
from bayes_opt import BayesianOptimization

from scipy.special import expit as sigmoid
from scipy.linalg.blas import dgemm
from scipy.stats import norm, gamma

In [ ]:
!pip3 install stable_baselines3
!pip3 install gymnasium

In [ ]:
import gymnasium as gym

In [ ]:
!apt install libgraphviz-dev
!pip install pygraphviz
import pygraphviz as pgv

In [ ]:
# @title

#from phi.jax.flow import *
# from phi.flow import *  # If JAX is not installed. You can use phi.torch or phi.tf as well.
#from phi.field._point_cloud import distribute_points
from tqdm.notebook import trange
from scipy import special

from typing import Tuple, Dict
import torch as pt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy import ndimage

from typing import Callable, Tuple, List, Optional, Union
from functools import partial


from scipy.optimize import NonlinearConstraint
import pymc as pm
import arviz as az
from scipy.integrate import solve_ivp

from scipy.stats import norm

import csv
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd


from scipy.fft import fft, ifft
from scipy.special import expit as sigmoid
from scipy.stats import norm

from typing import Callable, Optional, Union,Tuple,Any
from pymc import Model, Normal, sample, traceplot
from scipy.stats import norm


from scipy.optimize import minimize
from bayes_opt import BayesianOptimization
from matplotlib import gridspec


import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

from scipy.stats import norm
from scipy.interpolate import interp1d
from scipy.stats import norm
from scipy import ndimage
from scipy.sparse.linalg import cg


import scipy.sparse.linalg as splinalg
from scipy import interpolate
from scipy.sparse import diags


#from mpi4py import MPI
from scipy.sparse.linalg import LinearOperator, cg
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import laplace
from scipy.optimize import minimize
from scipy.interpolate import griddata

import arviz as az
%load_ext autoreload
%autoreload 2

seed = 42
az.style.use(['arviz-white', 'arviz-plasmish'])

from scipy.special import expit as sigmoid
from scipy.linalg.blas import saxpy
from scipy.stats import norm

from matplotlib import pyplot, cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp


# Define grid parameters
Lx = Ly = Lz = 1.0
Nx = Ny = Nz = 32
dx = dy = dz = Lx / Nx

# Create 3D grid
x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
z = np.linspace(0, Lz, Nz)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Initialize arrays
u = np.zeros((Ny, Nx, Nz))
v = np.zeros((Ny, Nx, Nz))
w = np.zeros((Ny, Nx, Nz))
p = np.zeros((Ny, Nx, Nz))
f = np.zeros((Ny, Nx, Nz))


# Set time parameters
dt = 0.001
t_end = 10.0
t = 0.0


S = 100  # Initial stock price
K = 125  # Strike price
T = 3  # Time to expiration in years
r = 0.05  # Risk-free interest rate
sigma = 0.9  # Volatility
dt = 2  # Time step for simulation
initial_portfolio_value = 150000  # Initial portfolio value
Ts = [40, 50,60,80]  # Time horizons for the rough volatility model

from scipy.stats import expon
U = 1000  # Initial capital
c = 500   # Premium rate
claim_frequency = 0.05  # Claim frequency
F = expon(scale=100)  # Exponential distribution for claim sizes
T_max = 1000000     # Maximum simulation time
jump_max = 50       # Maximum number of jumps


spot_prices = [100, 105,130]
strike_prices = [102, 103,200]
times_to_maturity = [0.5, 1.0, 1.5]
interest_rates = [0.02, 0.03, 0.04]
risk_free_rates = [0.02, 0.03,0.4]
volatilities = [0.20, 0.25,0.7]

num_particles=10000


P = 100  # Initial price
k = 100000  # Constant parameter
alpha = 0.001  # Alpha parameter
mu = 0.02  # Market impact coefficient
theta = 0.005  # Limit order fill rate
beta = 0.002  # Beta parameter
gamma = 0.1  # Gamma parameter
sigma = 0.03  # Volatility
v=0.04


a = 0.02  # Rate of mean reversion
b = 0.05  # Long-term mean
sigma = 0.01  # Volatility
r0 = 0.03  # Initial interest rate
T = 3     # Time horizon
N = 10000  # Number of steps


class PoisoningAttackCleanLabelBackdoor:
    """
    This class implements a poisoning attack with a clean label backdoor.
    """

    def __init__(
        self,
        trigger_func: Callable,
        target_label: Union[int, str, np.ndarray],
        dirty_label: Union[int, str, np.ndarray],
        flip_prob: float = 0.5,
        trigger_alpha: float = 0.5,
        poison_rate: float = 0.1,
        backdoor_trigger: Optional[Union[int, str, np.ndarray]] = None,
        backdoor_target: Optional[Union[int, str, np.ndarray]] = None,
        training_dataset: Optional[np.ndarray] = None,
        training_params: Optional[dict] = None,
        prior_mean: float = 0,
        prior_std: float = 1
    ) -> None:
        """
        Initialize the PoisoningAttackCleanLabelBackdoor instance.
        """
        self.trigger_func = trigger_func
        self.target_label = target_label
        self.dirty_label = dirty_label
        self.flip_prob = flip_prob
        self.trigger_alpha = trigger_alpha
        self.poison_rate = poison_rate
        self.backdoor_trigger = backdoor_trigger if backdoor_trigger is not None else 0
        self.backdoor_target = backdoor_target
        self.training_dataset = training_dataset
        self.training_params = training_params
        self.prior_mean = prior_mean
        self.prior_std = prior_std
        self.prior = np.random.normal(prior_mean, prior_std)


    def poison(
        self,
        x_audio: np.ndarray,
        y: Optional[np.ndarray] = None,
        broadcast: bool = False,
        random_seed: Optional[int] = None
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Apply the poisoning attack to the given audio data.
        """
        if y is None or not np.any(np.isin(self.target_label, y)):
            return x_audio, y

        num_poison = int(len(x_audio) * self.poison_rate)

        poisoned_labels = np.full((num_poison,), self.dirty_label)

        if broadcast:
            y_attack = np.broadcast_to(y, (x_audio.shape[0], y.shape[0]))
        else:
            y_attack = np.copy(y)

        np.random.seed(random_seed)

        for i in range(num_poison):
            trigger_pattern = self.trigger_func(x_audio[i])

            if np.random.rand() < self.flip_prob:
                poisoned_labels[i] = self.target_label[0]

            x_audio[i] = (1 - self.trigger_alpha) * x_audio[i] + self.trigger_alpha * trigger_pattern

            # Ensure S0 is a 1D array before passing it to simulate_rough_volatility_paths
            S0 = np.array([self.prior])  # Assuming self.prior is the intended S0 value


            # Call simulate_rough_volatility_paths with the corrected S0
            paths=self.simulate_rough_volatility_paths(Ts=Ts, S0=self.prior, sigma=0.8, gamma=0.7, dt=0.2)

        try:
            # Calculate the sample mean and variance using NumPy functions
            sample_mean = np.mean(x_audio)
            sample_variance = np.var(x_audio)

            # Update the prior with the sample statistics
            self.prior = np.random.normal(sample_mean, np.sqrt(sample_variance))


            # Adjusted code to unpack three values returned by _bayesian_sampling_diffusion_model
            trace_CIR, model_type_CIR = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.3, 3.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.4, 4.0, 10),
            beta=np.linspace(0.5, 5.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.cir_drift,  # Specify the non_linear_drift function
            model_type='CIR',

              )


            trace_libor_market, model_type_libor_market = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.3, 3.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.4, 4.0, 10),
            beta=np.linspace(0.5, 5.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.libor_market_drift,  # Specify the non_linear_drift function
            model_type='libor_market',

              )

            trace_vasicek, model_type_vasicek = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.4, 4.0, 10),
            alpha=np.linspace(0.3, 3.0, 10),
            beta=np.linspace(0.2, 2.0, 10),
            sigma=np.linspace(0.3, 3.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.vasicek_drift,  # Specify the non_linear_drift function
            model_type='vasicek'
             )

            trace_hull_white, model_type_hull_white = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.5, 5.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.3, 3.0, 10),
            beta=np.linspace(0.4, 4.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.hull_white_drift,  # Specify the non_linear_drift function
            model_type='hull_white'
              )

            trace_longstaff_schwartz, model_longstaff_schwartz = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.3, 3.0, 10),
            beta=np.linspace(0.4, 4.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.longstaff_schwartz_drift,  # Specify the non_linear_drift function
            model_type='longstaff_schwartz'
             )


        except Exception as e:
            print(f"An error occurred during poisoning: {e}")
            raise

        return x_audio, poisoned_labels


        #  libor market drift function
    def libor_market_drift(self, x, t, theta, mu, sigma):
        """
        Calculates the drift component of the LIBOR Market Model.

        Parameters:
        - x (float): Current level of the forward rate.
        - t (float): Time, typically measured in years.
        - theta (float): Long-term mean of the forward rate.
        - mu (float): Reversion speed towards the mean.
        - sigma (float): Volatility parameter.

        Returns:
        - float: Drift term calculated according to the LIBOR Market Model.

         Drift function for the LIBOR Market Model.
        """
        return theta * (mu - x) +  sigma * np.sqrt(t) * np.random.normal()



    #  Vasicek drift function
    def vasicek_drift(self, x, t, theta, mu, sigma):
        v = 1 / theta * (1 - np.exp(-theta * t))
        return theta * (mu - x) + sigma * np.sqrt(v) * np.random.normal()


     #The Cox-Ingersoll-Ross (CIR) model

    def cir_drift(self,alpha_c, t, theta, mu, sigma):
        return mu * (theta - alpha) + sigma * np.sqrt(alpha)

    def CIR_model(self,theta_c, mu_c, sigma_c, r0_c, T, N):
        dt = T / N
        r = np.zeros(N + 1)
        r[0] = r0_c
        # Initialize alpha_c as an array
        alpha_c = np.zeros(N + 1)
        alpha_c[0] = r0_c

        for i in range(1, N + 1):
              alpha_c[i] = alpha_c[i-1] + self.cir_drift(r[i-1], i*dt, theta_c, mu_c, sigma_c) * dt + \
                      sigma_c * np.sqrt(alpha_c[i-1]) * np.sqrt(dt) * np.random.normal(0, 1)
              r[i] = alpha_c[i]  # Update r with the new alpha_c value

        return r


    #  Hull-White drift function
    def hull_white_drift(self, x, t, theta, mu, sigma):
        v = 1 / theta * (1 - np.exp(-theta * t))
        phi_t = mu - theta * (x - mu) + sigma * np.sqrt(v) * np.random.normal()
        return theta * (mu - x) + phi_t


    def hull_white_volatility(self,t, theta_shw, sigma_shw):
        return sigma_shw * np.sqrt(1 / theta_shw * (1 - np.exp(-theta_shw * t)))

    def simulate_hull_white(self,r0_shw, theta_shw, mu_shw, sigma_shw, T, N=252):
        dt = T / N
        t = np.linspace(0, T, N+1)

        r = np.zeros(N+1)
        r[0] = r0_shw

        for i in range(1, N+1):
            dr = self.hull_white_drift(r[i-1], t[i], theta_shw, mu_shw, sigma_shw)
            dv = self.hull_white_volatility(t[i], theta_shw, sigma_shw)
            r[i] = r[i-1] + dr * dt + dv * np.sqrt(dt) * np.random.normal()

        return t, r


    def simulate_rough_volatility_paths(self, Ts, S0, sigma, gamma, dt=2):
        """
        Simulate paths of a rough volatility model for multiple time horizons.

        Parameters:
       - Ts: List of time horizons for simulation.
       - S0: Initial state vector.
       - sigma: Volatility parameter. Can be a scalar or a 1D array.
       - gamma: Parameter controlling the roughness of the volatility.
       - dt: Time step size.

       Returns:
       - paths: Simulated paths for each time horizon.
       """

       # Ensure S0 is a 1D array or a scalar
        if isinstance(S0, (float, np.float64, np.float32)):
          S0 = np.array([S0])  # Convert scalar to 1D array

        # Ensure S0 is a 1D array
        if not isinstance(S0, np.ndarray) or S0.ndim != 1:
            raise ValueError("Expected S0 to be a 1D numpy array")

        N = len(S0)
        all_paths = []


        for T in Ts:
           paths = np.zeros((int(T/dt), N))
           paths[0] = S0


           for t in range(1, int(T/dt)):
              dW = np.random.normal(size=N)
              dX = np.sqrt(dt) * (gamma * paths[t-1] + sigma * dW)
              paths[t] = paths[t-1] + dX


              all_paths.append(paths)


        return all_paths



    #  Longstaff-Schwartz drift function

    def longstaff_schwartz_drift(self, x, t, theta, mu, sigma):
        v = 1 / theta * (1 - np.exp(-theta * t))
        adjusted_drift = theta * (mu - x) + sigma * np.sqrt(v) * np.random.normal()
        adjusted_drift += mu - theta * (x - mu) + sigma * np.sqrt(v) * np.random.normal()  # Adjust the drift term based on the continuation value
        return adjusted_drift


    def optimal_transport(self, x_T, t, theta, mu, sigma):
        """
        Calculate the deterministic movement (transport component) based on the state and time.

        Parameters:
        - x_T: Current state at time T.
        - t: Time.
        - theta: Parameter for the drift function.
        - mu: Mean of the drift function.
        - sigma: Standard deviation of the drift function.

        Returns:
        - Deterministic movement based on the state and time.
        """

        return theta * (mu - x_T) + sigma * np.sqrt(1 / theta * (1 - np.exp(-theta * t))) * np.random.normal()



    def optimize_with_bayesian_optimization(self, objective_function, bounds, n_iter=50):
        """
        Performs Bayesian optimization over the given objective function.

        :param objective_function: Objective function to optimize.
        :param bounds: Dictionary specifying the lower and upper bounds for each parameter.
        :param n_iter: Number of iterations for the optimization.
        :return: Optimized parameters and the best observed value.
        """
        optimizer = BayesianOptimization(
            f=objective_function,
            pbounds=bounds,
            random_state=42,
            verbose=2,
        )

        optimizer.maximize(init_points=2, n_iter=n_iter)

        return optimizer.max['params'], optimizer.max['target']



    def objective_function(self, *args, **kwargs):
       # Extract parameters from kwargs since BayesianOptimization passes them as keyword arguments
       x, y = kwargs.get('x'), kwargs.get('y')

       # This could involve calling _bayesian_sampling_diffusion_model or another relevant function
       return -(x**2 + y**2 + 0.1*x*y + np.sin(x) + np.cos(y)) #-(x**2 + y**2)  # Minimize this function

       bounds = {'x': (-5, 5), 'y': (-5, 5)}

       # Optimize the objective function using Bayesian optimization

       optimized_params, best_value = self.optimize_with_bayesian_optimization(objective_function, bounds)
       constraint_limit = 0.5
       constraint = NonlinearConstraint(objective_function, -np.inf, constraint_limit)
       plot_constrained_opt(bounds, objective_function, optimizer);


       # Plot the optimization process
       optimizer = BayesianOptimization(
          f=objective_function,
          pbounds=bounds,
          random_state=42,
          verbose=2,
      )

       bounds_str = ', '.join([f'{k}={v[0]}:{v[1]}' for k, v in bounds.items()])
       print(f"Performing Bayesian optimization with bounds: {bounds_str}")
       optimized_params, best_value = self.optimize_with_bayesian_optimization(objective_function, bounds)

        # Prepare data for plotting
       data = pd.DataFrame({
            'Iteration': range(1, len(optimized_params) + 1),
            'Best Value': [best_value] * len(optimized_params),
            **{param: val for param, val in optimized_params.items()}
        })

        # Interactive plot using Plotly
       fig = make_subplots(rows=1, cols=1)
       fig.add_trace(go.Scatter(x=data['Iteration'], y=data['Best Value'], mode='lines+markers', name='Best Value'), row=1, col=1)
       for param in optimized_params:
            fig.add_trace(go.Scatter(x=data['Iteration'], y=data[param], mode='lines+markers', name=param), row=1, col=1)

       fig.update_layout(title_text=f'Optimization Process for Bounds: {bounds_str}', autosize=False,
                          width=500, height=400,
                          margin=dict(l=50, r=50, b=100, t=100, pad=4))

       # Show the interactive plot
       fig.show()


    def plot_optimization_process(self,bounds, filename='optimization_results.png'):
        optimizer = BayesianOptimization(
            f=self.objective_function,
            pbounds={'x': (-5, 5), 'y': (-5, 5)},
            random_state=42,
            verbose=2
        )

        # Perform the optimization
        optimizer.maximize(init_points=2, n_iter=50)
        acquisition_function = UtilityFunction(kind="ucb", kappa=0.1)
        optimizer.maximize(n_iter=10, acquisition_function=acquisition_function)
        optimized_params = optimizer.max['params']
        best_value = optimizer.max['target']
        print(f"Optimized Parameters: {optimized_params}, Best Value: {best_value}")

        bounds_str = ', '.join([f'{k}={v[0]}:{v[1]}' for k, v in bounds.items()])
        print(f"Performing Bayesian optimization with bounds: {bounds_str}")
        #optimized_params, best_value = self.optimize_with_bayesian_optimization(objective_function, bounds)

        # Prepare data for plotting
        data = pd.DataFrame({
            'Iteration': range(1, len(optimized_params) + 1),
            'Best Value': [best_value] * len(optimized_params),
            **{param: val for param, val in optimized_params.items()}
         })

        # Interactive plot using Plotly
        fig = make_subplots(rows=1, cols=1)
        fig.add_trace(go.Scatter(x=data['Iteration'], y=data['Best Value'], mode='lines+markers', name='Best Value'), row=1, col=1)
        for param in optimized_params:
            fig.add_trace(go.Scatter(x=data['Iteration'], y=data[param], mode='lines+markers', name=param), row=1, col=1)

        fig.update_layout(title_text=f'Optimization Process for Bounds: {bounds_str}', autosize=False,
                          width=500, height=400,
                          margin=dict(l=50, r=50, b=100, t=100, pad=4))

        # Show the interactive plot
        fig.show()



    def black_scholes_merton_call(self, S, K, T, r, sigma):
        """
        Calculate the price of a European call option using the BSM model.
        """

        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        call_price = (S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))


        # Check for NaN values and replace them
        call_price = np.nan_to_num(call_price, nan=0.0)


        return call_price



        def black_scholes_merton_put(self, S, K, T, r, sigma):
            """
            Calculate the price of a European put option using the BSM model.

            Parameters:
            S (float): Current stock price
            K (float): Strike price
            T (float): Time to expiration (years)
            r (float): Risk-free interest rate
            sigma (float): Volatility

            Returns:
            float: Put option price
            """
            # Convert to call price

            call_price = self.black_scholes_merton_call(S, K, T, r, sigma)
            put_price = K * np.exp(-r * T) - call_price

            return put_price



    def calculate_greeks(self, S, K, T, r, sigma):
        """
        Calculate the Greeks (delta, gamma, theta, vega, rho) of a European call option.
        """

        # Calculate Greeks
        call_price = self.black_scholes_merton_call(S, K, T, r, sigma)
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)


        delta = norm.cdf(d1)
        gamma = (norm.pdf(d1) / (S * sigma * np.sqrt(T)))
        theta = -0.5 * sigma**2 * S * norm.pdf(d1) * np.sqrt(T) - r * K * np.exp(-r * T) * norm.cdf(d2)


        # Vega calculation
        vega = S * np.sqrt(T) * norm.pdf(d1)


        # Rho calculation
        rho = K * np.exp(-r * T) * norm.cdf(d2)


        # Check for NaN values and replace them
        delta = np.nan_to_num(delta, nan=0.0)
        gamma = np.nan_to_num(gamma, nan=0.0)
        theta = np.nan_to_num(theta, nan=0.0)
        vega = np.nan_to_num(vega, nan=0.0)
        rho = np.nan_to_num(rho, nan=0.0)


        return delta, gamma, theta, vega, rho

    def dynamic_hedging(self, initial_portfolio_value, S, K, T, r, sigma, dt):
        """
        Implement dynamic hedging logic based on calculated Greeks.
        """

        # Initialize Greeks
        delta, gamma, theta, vega, rho = self.calculate_greeks(S, K, T, r, sigma)


        # Simulate price movements and apply hedging adjustments

        for t in range(int(T/dt)):
            # Simulate new price movement

            S_new = S + np.random.normal(0, sigma*np.sqrt(dt)) * S


            # Calculate new Greeks
            delta_new, gamma_new, theta_new, vega_new, rho_new = self.calculate_greeks(S_new, K, T, r, sigma)


            # Check for NaN values and replace them
            delta_new = np.nan_to_num(delta_new, nan=0.0)
            gamma_new = np.nan_to_num(gamma_new, nan=0.0)
            theta_new = np.nan_to_num(theta_new, nan=0.0)
            vega_new = np.nan_to_num(vega_new, nan=0.0)
            rho_new = np.nan_to_num(rho_new, nan=0.0)

            # Apply hedging adjustments
            portfolio_adjustment = delta_new * (S_new - S) + gamma_new * (S_new - S)**2
            initial_portfolio_value += portfolio_adjustment



            # Update portfolio value
            S = S_new

        return initial_portfolio_value



    def tail_probabilities_and_shortfall_integrals(self, S, K, T, r, sigma, threshold=5):
        """
        Calculate tail probabilities and shortfall integrals for a call option.

         Parameters:
         S (float): Underlying asset price
         K (float): Strike price
         T (float): Time to expiration
         r (float): Risk-free interest rate
         sigma (float): Volatility of the underlying asset
         threshold (float): Threshold value for shortfall calculation

         Returns:
        tuple: (tail_probability, shortfall_integral)
        """
        # Calculate d1 and d2
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)


        # Calculate tail probability
        tail_probability = 1 - norm.cdf(d2)


        # Calculate shortfall integral
        shortfall_integral = (K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)) - threshold


        # Handle potential numerical issues
        tail_probability = np.clip(tail_probability, 0, 1)
        shortfall_integral = np.clip(shortfall_integral, 0, None)


        # Calculate VaR for 95% and 99% confidence levels
        var_95 = norm.ppf(0.05, d1, sigma * np.sqrt(T))
        var_99 = norm.ppf(0.01, d1, sigma * np.sqrt(T))


        # Convert VaR values to percentage loss
        var_95_percent = (1 - np.exp(var_95)) * 100
        var_99_percent = (1 - np.exp(var_99)) * 100


        # Check for NaN values and replace them
        var_95_percent = np.nan_to_num(var_95_percent, nan=0.0)
        var_99_percent = np.nan_to_num(var_99_percent, nan=0.0)
        shortfall_integral = np.nan_to_num(shortfall_integral, nan=0.0)
        tail_probability = np.nan_to_num(tail_probability, nan=0.0)



        return tail_probability, shortfall_integral, var_95_percent, var_99_percent


    def fourier_transform_option_price(self,S, K, T, r, sigma, alpha=0.2, beta=0.4, max_iter=1000):
        """
        Calculate the price of a European call option using the Fourier transform method.

        Parameters:
        - S: Spot price of the underlying asset.
        - K: Strike price of the option.
        - T: Time to maturity in years.
        - r: Risk-free interest rate.
        - sigma: Volatility of the underlying asset.
        - alpha: Smoothing parameter for numerical stability.
        - beta: Parameter controlling the width of the Fourier inversion window.
        - max_iter: Maximum number of iterations for the optimization routine.

         Returns:
        - Option price calculated using the Fourier transform method.
        """
        def characteristic_function(u):
            """
            Characteristic function of the log-return distribution.
            """
            mu = (r - 0.5 * sigma ** 2) * u
            phi = np.exp(mu) / (u ** 2 + sigma ** 2)
            return np.exp(1j * mu) * phi

        def inverse_fourier_transform(u, C):
            """
            Inverse Fourier transform to calculate the option price.
            """
            w = np.linspace(-beta, beta, len(C)) + 1j * alpha
            w = np.fft.fftshift(w)
            C = np.fft.fftshift(C)
            C_hat = np.fft.fft(C)

            C_hat = np.fft.fftshift(np.fft.fft(C - 1j * w))

            P = np.real(np.fft.ifft(C_hat))
            P = np.fft.fftshift(P)
            P = np.fft.ifft(P)
            P = np.fft.fftshift(P)

            return P[0]

        # Generate a range of frequencies for the Fourier transform
        u = np.linspace(-10, 10, 1000)
        u = np.fft.fftshift(u)

        # Calculate the characteristic function at these frequencies
        C = np.array([characteristic_function(ui) for ui in u])

        # Perform the inverse Fourier transform to get the option price
        option_price = inverse_fourier_transform(u, C)
        print(f"Option Price: {option_price}")
        option_price = option_price.real


        # Ensure the result is real-valued and handle edge cases
        option_price = np.real(option_price).item()
        if np.isnan(option_price):
            option_price = 0.0  # Default value if calculation fails

        return option_price



    def display_results_for_multiple_parameters(self, spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities):
        """
        Display the option prices for multiple sets of parameters.

        Parameters:
        - spot_prices: List of spot prices.
        - strike_prices: List of strike prices.
        - times_to_maturity: List of time to maturities in years.
        - risk_free_rates: List of risk-free interest rates.
        - volatilities: List of volatilities.
        """
        for S in spot_prices:
            for K in strike_prices:
                for T in times_to_maturity:
                    for r in risk_free_rates:
                        for sigma in volatilities:
                            option_price = self.fourier_transform_option_price(S, K, T, r, sigma)
                            print(f"S={S}, K={K}, T={T}, r={r}, sigma={sigma}, Option Price: {option_price}")


    def collect_and_plot_option_prices(self, spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities):
        """
        Collect option prices for multiple sets of parameters and plot them.
        """
        # Initialize empty lists to store option prices
        prices = []

        # Iterate over all combinations of parameters
        for S in spot_prices:
            for K in strike_prices:
                for T in times_to_maturity:
                    for r in risk_free_rates:
                        for sigma in volatilities:
                            option_price = self.fourier_transform_option_price(S, K, T, r, sigma)
                            print(f"S={S}, K={K}, T={T}, r={r}, sigma={sigma}, Option Price: {option_price}")
                            # Append the results to the list
                            prices.append((S, K, T, r, sigma, option_price))


        # Convert the list of tuples into a DataFrame for easier manipulation
        df = pd.DataFrame(prices, columns=['Spot Price', 'Strike Price', 'Time to Maturity', 'Risk-Free Rate', 'Volatility', 'Option Price'])


        # Save the DataFrame to a CSV file
        pdf = pd.DataFrame(df)
        pdf.to_csv('option_prices.csv', index=False)
        print(df)



    def cramler_lundberg_model(self,U, c, claim_frequency, F, T_max, jump_max):
        # Generate claim frequencies and sizes
        claim_freq = np.random.poisson(claim_frequency, T_max)
        claim_sizes = np.random.choice(F.rvs().flatten(), size=claim_freq.sum())
        claim_sizes.sort()


        claim_sizes = np.diff(np.insert(claim_sizes, 0, 0))
        claim_sizes = claim_sizes[claim_sizes > 0]
        claim_sizes = np.random.choice(claim_sizes, size=T_max)
        claim_sizes = np.cumsum(claim_sizes)


        # Initialize surplus and time

        surplus = U
        time = 0

        # Initialize lists to store surplus values
        surplus_values = []

        # Simulate until ruin or max time reached
        while surplus > 0 and time < T_max:

           # Add premium
           surplus += c


           # Subtract claims
           claim_amount = np.sum(claim_sizes[:min(time+1, len(claim_sizes))])


           surplus -= claim_amount
           # Ensure surplus doesn't go below zero
           surplus = max(0, surplus - claim_amount)  # Ensure surplus doesn't go below zero


           # Check for capital withdrawals (negative jumps)

           if np.random.rand() < 0.01:  # 1% chance of positive jump
               surplus += np.random.exponential(scale=10)


           # Update time and reset claim sizes

           time += 1
           claim_sizes = claim_sizes[min(time, len(claim_sizes)):]

           # Store current surplus value
           surplus_values.append(surplus)

        return surplus_values



    def buhlmann_model(self,iota, phi, zeta, lambda_, omega):
        """
        Buhlmann model for credibility theory

        Parameters:
        iota (float): Prior mean of the underlying distribution
        phi (float): Prior variance of the underlying distribution
        zeta (float): Number of periods of experience
        lambda_ (float): Underlying distribution parameter
        mu (float): Mean of the underlying distribution

        Returns:
        tuple: Predicted mean and standard deviation
        """
        # Calculate the credibility factor
        kappa = (phi / (phi + zeta * omega**2))


        # Calculate the time-dependent credibility factor
        kappa_t = kappa * np.exp(-gamma * t)


        # Calculate the adjusted credibility factor
        kappa_adj = kappa_t * (1 - alpha) + alpha


        # Calculate the premium loading
        premium_loading = beta * omega

        # Calculate the risk loading
        risk_loading = alpha * omega


        # Calculate the predicted mean and standard deviation
        predicted_mean = iota + kappa * omega * (lambda_ - iota)
        predicted_std = np.sqrt(phi * (1 - kappa))


        # Calculate the credibility factor
        credibility_factor = kappa_adj * (1 - alpha) + alpha


        return predicted_mean, predicted_std

    def mean_field_interaction(self,x):
        """
        Compute the mean field interaction for the McKean-Vlasov process.

        Parameters:
        x (numpy array): Current state of the process

        Returns:
        float: Mean field interaction value
        """
        # Use np.mean() for efficiency
        mean_field_effect = np.mean(x)


        # Add a small constant to avoid division by zero
        epsilon = 1e-8

        # Compute the squared difference between each element and the mean
        squared_diff = (x - mean_field_effect)**2


        # Compute the mean of the squared differences
        var_x = np.mean(squared_diff)


        # Compute the second-order correction term
        second_order_correction = 0.5 * var_x


        # Combine terms
        drift_term = mean_field_effect + second_order_correction

        return drift_term


    def mckean_vlasov_process(self,P0, mu, sigma, t_max, dt, num_steps):
        """
        Simulate the McKean-Vlasov process.

        Parameters:
        S0 (float): Initial value of the process
        mu (callable): Drift term function
        sigma (float): Volatility
        t_max (float): Maximum time
        dt (float): Time step size
        num_steps (int): Number of steps

        Returns:
        numpy array: Array of simulated values at each time step
        """

        n = int(t_max / dt)
        t = np.linspace(0, t_max, n+1)
        # Initialize the process
        P = np.zeros(n+1)
        P[0] = P0



        # Generate random numbers for the Brownian motion
        # Simulate the process
        for i in range(1, n+1):
            dW = norm.rvs(size=1)
            P[i] = P[i-1] * np.exp((mu(P[i-1]) - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * dW)


        return t, P




    def plot_results(self):
        # Assuming S, K, T, r, sigma, and dt are defined elsewhere in your class
        # Generate time steps for plotting
        timesteps = np.linspace(0, T, num=100)


        #model_type = 'CIR'
        for model_type in ['vasicek', 'hull_white', 'longstaff_schwartz', 'libor_market', 'CIR']:
            model_type = "vasicek";
            model_type= "hull_white";
            model_type= "longstaff_schwartz";
            model_type= "libor_market";
            model_type= "CIR";



        # Define a base filename for the plots
        base_filename = f"{model_type}_results"


        # Plot Call Price
        plt.figure(figsize=(10, 6))
        plt.plot(timesteps, self.black_scholes_merton_call(S, K, timesteps, r, sigma), label='Call Price')
        plt.xlabel('Time')
        plt.ylabel('Price')
        plt.title(f'Evolution of Call Option Price Over Time ({model_type})')
        plt.savefig(f'{base_filename}_call_price_plot.png', dpi=300, bbox_inches='tight')
        plt.legend()
        plt.show()

        # Plot Greeks
        plt.figure(figsize=(10, 6))
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[0], label='Delta')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[1], label='Gamma')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[2], label='Theta')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[3], label='Vega')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[4], label='Rho')
        plt.xlabel('Time')
        plt.ylabel('Greeks')
        plt.title(f'Evolution of Greeks Over Time ({model_type})')
        plt.savefig(f'{base_filename}_greeks_plot.png', dpi=300, bbox_inches='tight')
        plt.legend()
        plt.show()



    def plot_constrained_opt(bounds, objective_function, optimizer):
        """
        Plots a number of interesting contours to visualize constrained 2-dimensional optimization.
        """

        # Set a few parameters
        n_constraints = optimizer.constraint.lb.size
        n_plots_per_row = 2+n_constraints

        # Construct the subplot titles
        if n_constraints==1:
          c_labels = ["constraint"]
        else:
           c_labels = [f"constraint {i+1}" for i in range(n_constraints)]
        labels_top = ["target"] + c_labels + ["masked target"]
        labels_bot = ["target estimate"] + [c + " estimate" for c in c_labels] + ["acquisition function"]
        labels = [labels_top, labels_bot]

        # Setup the grid to plot on
        x = np.linspace(bounds['x'][0], bounds['x'][1], 1000)
        y = np.linspace(bounds['y'][0], bounds['y'][1], 1000)
        xy = np.array([[x_i, y_j] for y_j in y for x_i in x])
        X, Y = np.meshgrid(x, y)

        # Evaluate the actual functions on the grid
        Z = objective_function(X, Y)
        # This reshaping is a bit painful admittedly, but it's a consequence of np.meshgrid
        C = optimizer.constraint.fun(X, Y).reshape((n_constraints,) + Z.shape).swapaxes(0, -1)


        fig, axs = plt.subplots(2, n_plots_per_row, constrained_layout=True, figsize=(12,8))

        for i in range(2):
           for j in range(n_plots_per_row):
              axs[i, j].set_aspect("equal")
              axs[i, j].set_title(labels[i][j])


        # Extract & unpack the optimization results
        max_ = optimizer.max
        res = optimizer.res
        x_ = np.array([r["params"]['x'] for r in res])
        y_ = np.array([r["params"]['y'] for r in res])
        c_ = np.array([r["constraint"] for r in res])
        a_ = np.array([r["allowed"] for r in res])


        Z_est = optimizer._gp.predict(xy).reshape(Z.shape)
        C_est = optimizer.constraint.approx(xy).reshape(Z.shape + (n_constraints,))
        P_allowed = optimizer.constraint.predict(xy).reshape(Z.shape)

        Acq = np.where(Z_est >0, Z_est * P_allowed, Z_est / (0.5 + P_allowed))


        target_vbounds = np.min([Z, Z_est]), np.max([Z, Z_est])
        constraint_vbounds = np.min([C, C_est]), np.max([C, C_est])


        axs[0,0].contourf(X, Y, Z, cmap=plt.cm.coolwarm, vmin=target_vbounds[0], vmax=target_vbounds[1])
        for i in range(n_constraints):
          axs[0,1+i].contourf(X, Y, C[:,:,i], cmap=plt.cm.coolwarm, vmin=constraint_vbounds[0], vmax=constraint_vbounds[1])
        Z_mask = Z

        Z_mask[~np.squeeze(optimizer.constraint.allowed(C))] = np.nan
        axs[0,n_plots_per_row-1].contourf(X, Y, Z_mask, cmap=plt.cm.coolwarm, vmin=target_vbounds[0], vmax=target_vbounds[1])

        axs[1,0].contourf(X, Y, Z_est, cmap=plt.cm.coolwarm, vmin=target_vbounds[0], vmax=target_vbounds[1])
        for i in range(n_constraints):
           axs[1,1+i].contourf(X, Y, C_est[:, :, i], cmap=plt.cm.coolwarm, vmin=constraint_vbounds[0], vmax=constraint_vbounds[1])
        axs[1,n_plots_per_row-1].contourf(X, Y, Acq, cmap=plt.cm.coolwarm, vmin=0, vmax=1)

        for i in range(2):
           for j in range(n_plots_per_row):
              axs[i,j].scatter(x_[a_], y_[a_], c='white', s=80, edgecolors='black')
              axs[i,j].scatter(x_[~a_], y_[~a_], c='red', s=80, edgecolors='black')
              axs[i,j].scatter(max_["params"]['x'], max_["params"]['y'], s=80, c='green', edgecolors='black')

        return fig, axs



    def constraint_function_2_dim(x, y):
        return np.array([- np.cos(x) * np.cos(y) + np.sin(x) * np.sin(y), - np.cos(x) * np.cos(-y) + np.sin(x) * np.sin(-y)])

    constraint_lower = np.array([-np.inf, -np.inf])
    constraint_upper = np.array([0.6, 0.6])



    def simulate_high_frequency_trading(self,prices, bid_ask_spreads, liquidity_factor):
        """
        Simulate a high-frequency trading strategy with the given market prices and bid-ask spreads.

        prices: Array of simulated prices
        bid_ask_spreads: Array of simulated bid-ask spreads
        liquidity_factor: Factor affecting market liquidity (lower values indicate less liquidity)

        Returns: Cumulative profit, cumulative slippage, and plot of results
        """
        n_trades = len(prices)
        slippage = []
        profits = []

        # Simulate trades at every price point
        for i in range(1, n_trades):
            # Simulated trade with price slippage due to bid-ask spread
            trade_price = prices[i] + np.random.uniform(-bid_ask_spreads[i], bid_ask_spreads[i])
            trade_price *= liquidity_factor  # Apply liquidity factor
            trade_price = np.clip(trade_price, prices[i] - bid_ask_spreads[i], prices[i] + bid_ask_spreads[i])


            # Calculate slippage (difference between expected and actual price)
            trade_slippage = abs(trade_price - prices[i])
            slippage.append(trade_slippage)


            # Profit from this trade (difference between buy and sell price)
            profit = prices[i] - prices[i - 1] - trade_slippage
            profits.append(profit)


            cumulative_profit = np.cumsum(profits)
            cumulative_slippage = np.cumsum(slippage)


            return cumulative_profit, cumulative_slippage


    def simulate_market_liquidity(self,prices, bid_ask_spreads, liquidity_factor):
        np.random.seed(42)
        n_trades = len(prices)
        prices = np.cumsum(np.random.randn(n_trades) * 0.5 + 100)
        bid_ask_spreads = np.random.uniform(0.001, 0.02 / liquidity_factor, size=n_trades)
        return prices, bid_ask_spreads

    def simulate_trade_execution(self,prices, bid_ask_spreads, liquidity_factor):
        n_trades = len(prices)
        slippage = []
        profits = []

        for i in range(1, n_trades):
            # Simulate trade with price slippage due to bid-ask spread
            trade_price = prices[i] + np.random.uniform(-bid_ask_spreads[i], bid_ask_spreads[i]) * liquidity_factor

            # Apply more sophisticated slippage calculation
            slippage_term = np.abs(trade_price - prices[i]) * np.exp(-(i/n_trades))  # Exponential decay of slippage
            trade_slippage = slippage_term

            # Calculate profit from this trade
            profit = prices[i] - prices[i-1] - trade_slippage

            # Apply stop-loss/take-profit logic
            if profit > 0 and np.random.rand() < 0.1:  # 10% chance of taking profit
                   profit = max(profit, profit * 0.9)  # Take 90% of profit
            elif profit < 0 and np.random.rand() < 0.1:  # 10% chance of cutting loss
                   profit = min(profit, profit * 0.9)  # Cut 90% of loss

                   slippage.append(trade_slippage)
                   profits.append(profit)

            return np.cumsum(profits), np.cumsum(slippage)



    def run_simulation(self,prices, bid_ask_spreads, liquidity_factor):
        try:
            # Simulate market liquidity
            prices_sim, bid_ask_spreads_sim = self.simulate_market_liquidity(prices, bid_ask_spreads, liquidity_factor)
            print(f"Simulated prices: {prices_sim}")
            print(f"Simulated bid-ask spreads: {bid_ask_spreads_sim}")

            # Simulate trade execution
            cumulative_profit, cumulative_slippage = self.simulate_trade_execution(prices_sim, bid_ask_spreads_sim, liquidity_factor)


            return cumulative_profit, cumulative_slippage
        except Exception as e:
            print(f"An error occurred: {str(e)}")
            return None, None, None



    def logistic_map(self,r_le, x):
          return r_le * x * (1 - x)

    def lyapunov_exponent(self,r_le, num_iterations=1000, num_discard=100):
          x = 0.5
          lyap = 0

          for _ in range(num_discard):
              x = self.logistic_map(r_le, x)

          for _ in range(num_iterations):
                 x = self.logistic_map(r_le, x)
                 lyap += np.log(abs(r_le * (1 - 2*x)))

          return lyap / num_iterations



    def bifurcation_diagram(self,r_min, r_max, num_r, num_iterations, num_discard):
        r_range = np.linspace(r_min, r_max, num_r)
        x = np.ones(num_r) * 0.5

        results = []
        for r in r_range:
            for _ in range(num_discard):
                    x = self.logistic_map(r, x)
            for _ in range(num_iterations):
                    x = self.logistic_map(r, x)
                    results.append((r, x))

        return np.array(results)



    def calculate_optimal_speed(self,S, t):
        return P / (T + k / alpha)

    def simulate_execution(self,num_steps=10000):
        dt = T / num_steps
        S = np.zeros(num_steps)
        Q_mkt = np.zeros(num_steps)
        Q_lim = np.zeros(num_steps)

        S[0] = 100  # Initial stock price
        Q_mkt[0] = P * mu / (mu + theta)
        Q_lim[0] = P * mu / (mu + beta)

        for i in range(1, num_steps):
            nu_star_mkt = self.calculate_optimal_speed(S[i-1], i*dt)
            nu_star_lim = nu_star_mkt * beta / mu

            # Market order execution
            dS_mkt = sigma * np.sqrt(dt) * np.random.normal() + theta * nu_star_mkt * dt
            dQ_mkt = -nu_star_mkt * dt

            S[i] += dS_mkt
            Q_mkt[i] = max(Q_mkt[i-1] + dQ_mkt, 0)

            # Limit order execution
            fill_rate = norm.cdf(nu_star_lim * gamma * np.sqrt(dt))
            dQ_lim = -nu_star_lim * dt * fill_rate

            Q_lim[i] = max(Q_lim[i-1] + dQ_lim, 0)

            # Update total liquidated shares
            Q_total = Q_mkt[i] + Q_lim[i]

        return S, Q_mkt, Q_lim, Q_total




        def calculate_optimal_speed(self,S, t):
          return P / (T + k / alpha)

        def simulate_execution(self,num_steps=10000):
            dt = T / num_steps
            S = np.zeros(num_steps)
            Q_mkt = np.zeros(num_steps)
            Q_lim = np.zeros(num_steps)

            S[0] = 100  # Initial stock price
            Q_mkt[0] = P * mu / (mu + theta)
            Q_lim[0] = P * mu / (mu + beta)

            for i in range(1, num_steps):
                nu_star_mkt = self.calculate_optimal_speed(S[i-1], i*dt)
                nu_star_lim = nu_star_mkt * beta / mu

                # Market order execution
                dS_mkt = sigma * np.sqrt(dt) * np.random.normal() + theta * nu_star_mkt * dt
                dQ_mkt = -nu_star_mkt * dt

                S[i] += dS_mkt
                Q_mkt[i] = max(Q_mkt[i-1] + dQ_mkt, 0)

                # Limit order execution
                fill_rate = norm.cdf(nu_star_lim * gamma * np.sqrt(dt))
                dQ_lim = -nu_star_lim * dt * fill_rate

                Q_lim[i] = max(Q_lim[i-1] + dQ_lim, 0)

                # Update total liquidated shares
                Q_total = Q_mkt[i] + Q_lim[i]

            return S, Q_mkt, Q_lim, Q_total

    def run_multiple_scenarios(self):
        num_steps_list = [5000, 10000, 20000]
        initial_price_list = [100, 150, 200]
        volatility_list = [0.2, 0.3, 0.4]
        drift_list = [0.05, 0.1, 0.15]
        theta_list = [0.01, 0.02, 0.03]
        beta_list = [0.005, 0.0075, 0.01]
        gamma_list = [0.5, 1.0, 1.5]
        alpha_list = [0.1, 0.2, 0.3]

        results = []
        for num_steps in num_steps_list:
            print(f"Running scenario with num_steps = {num_steps}")
            S, Q_mkt, Q_lim, Q_total = self.simulate_execution(num_steps)
            results.append((S, Q_mkt, Q_lim, Q_total))

        return results




    def initialize_system(self,Lx, Ly, Lz, Nx, Ny, Nz):
        dx = Lx / (Nx - 1)
        dy = Ly / (Ny - 1)
        dz = Lz / (Nz - 1)
        t = 0.0
        t_end = 1.0
        dt = 0.001


        u = np.random.rand(Nx, Ny, Nz)
        v = np.random.rand(Nx, Ny, Nz)
        w = np.random.rand(Nx, Ny, Nz)


        # Smooth initial velocity profile
        x, y, z = np.meshgrid(
             np.linspace(0, Lx, Nx),
             np.linspace(0, Ly, Ny),
             np.linspace(0, Lz, Nz)


               )

        r = np.sqrt(x**2 + y**2 + z**2)
        u[:] = np.sin(r) * np.exp(-r)
        v[:] = np.cos(r) * np.exp(-r)
        w[:] = np.sin(r) * np.exp(-r)


        # Set initial velocity profiles
        u[:, :, :] = 1.0 * (np.sin(np.pi * np.arange(Nx) / Nx) * np.cos(np.pi * np.arange(Ny) / Ny) *
                       np.exp(-(np.arange(Nz) / Nz)**2))
        v[:, :, :] = 0.5 * (np.sin(np.pi * np.arange(Ny) / Ny) * np.cos(np.pi * np.arange(Nx) / Nx) *
                       np.exp(-(np.arange(Nz) / Nz)**2))
        w[:, :, :] = 0.25 * (np.sin(np.pi * np.arange(Nz) / Nz) * np.cos(np.pi * np.arange(Nx) / Nx) *
                         np.exp(-(np.arange(Ny) / Ny)**2))


        return u, v, w, dx, dy, dz

    def navier_stokes(self,u, v, w, dx, dy, dz, dt, rho, mu, k, T, Pr=0.7, alpha=0.01, epsilon=0.1,epsilon_turb=0.1,Cepsilon1=1.44, Cepsilon2=1.92, Ckmu=0.09, Cp=1000):

        """
        Solve the Navier-Stokes equations using the stable fluids algorithm.

        Parameters:
        u, v, w (ndarray): Initial velocity components
        dx, dy, dz (float): Grid spacings
        dt (float): Time step
        rho (float): Fluid density
        mu (float): Dynamic viscosity
        alpha (float): Smoothing parameter (default: 0.01)
        beta (float): Projection parameter (default: 0.5)

        Returns:
        u_new, v_new, w_new (ndarray): Updated velocity components
        """

        Re = rho * dx * np.sqrt(u**2 + v**2 + w**2) / mu


        # Compute velocity gradients
        du_dx = (np.roll(u, -1, axis=1) - u) / dx
        dv_dy = (np.roll(v, -1, axis=0) - v) / dy
        dw_dz = (np.roll(w, -1, axis=2) - w) / dz



        # Compute strain rates
        e11 = du_dx
        e22 = dv_dy
        e33 = dw_dz
        e12 = (du_dx + np.roll(u, -1, axis=1) - 2*u) / (2*dx)
        e23 = (dv_dy + np.roll(v, -1, axis=0) - 2*v) / (2*dy)
        e31 = (dw_dz + np.roll(w, -1, axis=2) - 2*w) / (2*dz)


        # Compute viscous stresses
        tau11 = 2 * mu * e11
        tau22 = 2 * mu * e22
        tau33 = 2 * mu * e33
        tau12 = mu * e12
        tau23 = mu * e23
        tau31 = mu * e31



        # Compute pressure gradient
        grad_p = -(rho * (u * du_dx + v * dv_dy + w * dw_dz +
                       tau11 + tau22 + tau33 + tau12 + tau23 + tau31))



        # Compute divergence of velocity
        div_v = (np.sum(np.array([du_dx, dv_dy, dw_dz]), axis=0))


        # Compute nonlinear term
        nonlinear_term = div_v * np.array([u, v, w])
        nonlinear_term = nonlinear_term.reshape(3, Nx, Ny, Nz)
        nonlinear_term = np.sum(nonlinear_term, axis=0)

        nonlinear_term = nonlinear_term.sum(axis=0)
        nonlinear_term = -1.0 * nonlinear_term
        nonlinear_term = nonlinear_term * Re
        nonlinear_term = nonlinear_term * dt
        nonlinear_term = nonlinear_term / 2.0


        # Apply Laplacian smoothing
        laplacian_u = ndimage.laplace(u)
        laplacian_v = ndimage.laplace(v)
        laplacian_w = ndimage.laplace(w)



        # Update velocities with smoothing
        u_smooth = u - alpha * dt * laplacian_u
        v_smooth = v - alpha * dt * laplacian_v
        w_smooth = w - alpha * dt * laplacian_w


        # Update velocities
        u_new = u_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        v_new = v_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        w_new = w_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)


        # Update velocities
        u_new = u_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31 + nonlinear_term[0])
        v_new = v_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31 + nonlinear_term[1])
        w_new = w_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31 + nonlinear_term[2])


        # Periodic boundary conditions
        u[:, :, :] = np.roll(u, 1, axis=1)
        v[:, :, :] = np.roll(v, 1, axis=0)
        w[:, :, :] = np.roll(w, 1, axis=2)


        # Update velocities
        u_new = u - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        v_new = v - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        w_new = w - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)



        # Pressure correction
        pressure_correction = splinalg.cg(
            A=splinalg.LinearOperator(
               shape=(np.prod(u.shape), np.prod(u.shape)),
               matvec=lambda x: ndimage.laplace(x.reshape(u.shape)).flatten(),
                      ),
             b=np.zeros(np.prod(u.shape)),
             maxiter=None,
                )[0].reshape(u.shape)



        # Correct velocities

        u_new -= pressure_correction
        v_new -= pressure_correction
        w_new -= pressure_correction


        # Update velocities with smoothing
        u_smooth = u - alpha * dt * laplacian_u
        v_smooth = v - alpha * dt * laplacian_v
        w_smooth = w - alpha * dt * laplacian_w


        # Turbulence model (k-ε)

        T = np.random.rand(Nx, Ny, Nz) * 100
        k = np.random.rand(Nx, Ny, Nz) * 1e-6
        mu_turb = np.random.rand(Nx, Ny, Nz) * 1e-5

        epsilon = np.random.rand(Nx, Ny, Nz) * 1e-6
        rho = np.random.rand(Nx, Ny, Nz) * 1.0 + 1.0
        mu = np.random.rand(Nx, Ny, Nz) * 1e-5 + 1e-5
        Pr = 0.7



        k_new = k - dt * (mu_turb / rho * epsilon)
        epsilon_new = epsilon - dt * (
        Cepsilon1 * epsilon**2 / k +
        Cepsilon2 * epsilon / k * mu_turb / rho * epsilon
             )

        # Heat transfer equation
        T_new = T - dt * (
        Pr * mu / (rho * Cp) * ndimage.laplace(T)
                   )


        # Apply wall functions for boundary layers ; https://www.simscale.com/forum/t/what-is-y-yplus/82394
        u_wall = np.sqrt(mu_turb / rho * epsilon)
        v_wall = 0.0
        w_wall = 0.0
        T_wall = T_new[0]

        # Apply free surface condition
        # This is a simplification; actual implementation would involve solving for free surface height
        rho_free_surface = 1.225  # Air density https://en.wikipedia.org/wiki/Density_of_air
        mu_free_surface = 1.81e-5  # Air viscosity. https://en.wikipedia.org/wiki/Viscosity
        if np.any(rho < rho_free_surface):

            # Mix fluid properties at interface
            mix_rho = np.where(rho < rho_free_surface, rho, rho_free_surface)
            mix_mu = np.where(rho < rho_free_surface, mu, mu_free_surface)

        laplacian_k = ndimage.laplace(k)
        laplacian_epsilon = ndimage.laplace(epsilon)
        k_smooth = k - alpha * dt * laplacian_k
        epsilon_smooth = epsilon - alpha * dt * laplacian_epsilon


        # Update turbulence variables. https://en.wikipedia.org/wiki/Turbulence_modeling
        k_new = k_smooth + dt * (
        mu_turb / rho * epsilon +
        Ckmu * mu_turb / rho * epsilon**2 / k
            )
        epsilon_new = epsilon_smooth + dt * (
        Cepsilon1 * epsilon**2 / k * mu_turb / rho +
        Cepsilon2 * mu_turb / rho * epsilon**2 / k
             )


        # Pressure correction
        def pressure_correction():
            A = ndimage.laplace(np.stack([u_smooth, v_smooth, w_smooth], axis=2))
            b = div_v
            p, _ = cg(A, b)
            return p.reshape(u_smooth.shape)


        # Correct velocities
        u_new = u_smooth - dt * grad_p - dt * (tau11 + tau22 + tau33 + tau12 + tau23 + tau31) + epsilon * dt * div_v
        v_new = v_smooth - dt * grad_p - dt * (tau11 + tau22 + tau33 + tau12 + tau23 + tau31) + epsilon * dt * div_v
        w_new = w_smooth - dt * grad_p - dt * (tau11 + tau22 + tau33 + tau12 + tau23 + tau31) + epsilon * dt * div_v

        # Periodic boundary conditions
        u_new[:, :, :] = np.roll(u_new, 1, axis=1)
        v_new[:, :, :] = np.roll(v_new, 1, axis=0)
        w_new[:, :, :] = np.roll(w_new, 1, axis=2)


        # Compute convective acceleration terms
        convective_u = u * du_dx + v * dv_dy + w * dw_dz
        convective_v = u * du_dx + v * dv_dy + w * dw_dz
        convective_w = u * du_dx + v * dv_dy + w * dw_dz


        # Compute pressure gradient
        grad_p = -(rho * (convective_u + convective_v + convective_w +
                   tau11 + tau22 + tau33 + tau12 + tau23 + tau31))


        # Add terms representing the proof of existence and regularity
        laplacian_u = (du_dx + dv_dy + dw_dz)
        laplacian_v = (du_dx + dv_dy + dw_dz)
        laplacian_w = (du_dx + dv_dy + dw_dz)

        # Add regularization terms
        regularization_u = alpha * laplacian_u
        regularization_v = alpha * laplacian_v
        regularization_w = alpha * laplacian_w

        # Update velocity components
        u_new = u + dt * (-convective_u - tau11 + mu * laplacian_u + regularization_u)
        v_new = v + dt * (-convective_v - tau22 + mu * laplacian_v + regularization_v)
        w_new = w + dt * (-convective_w - tau33 + mu * laplacian_w + regularization_w)


        # Update temperature
        #T_new = T_new + pressure_correction


        # Compute turbulent dissipation rate
        epsilon_turb_new = epsilon_turb + alpha * (epsilon_turb / k - Cepsilon1 * epsilon_turb**2 / (Cepsilon2 * k + epsilon))


        # Update velocities using semi-Lagrangian advection https://physics.stackexchange.com/questions/356407/validity-of-the-navier-stokes-equations-for-turbulent-flows
        u_new = u + dt * (nonlinear_term[0] + mu * du_dx + epsilon_turb_new * u)
        v_new = v + dt * (nonlinear_term[1] + mu * dv_dy + epsilon_turb_new * v)
        w_new = w + dt * (nonlinear_term[2] + mu * dw_dz + epsilon_turb_new * w)


        # Update kinetic energy
        k_new = k + dt * (0.5 * (nonlinear_term[0]**2 + nonlinear_term[1]**2 + nonlinear_term[2]**2) / 3 - mu * (du_dx**2 + dv_dy**2 + dw_dz**2) - epsilon_turb)


        # Update temperature
        T_new = T + dt * (Cp * (nonlinear_term[0] * du_dx + nonlinear_term[1] * dv_dy + nonlinear_term[2] * dw_dz) / rho)


        # Update velocities
        u_new = u - dt * (nonlinear_term[0] + alpha * (grad_p[0] / dx))
        v_new = v - dt * (nonlinear_term[1] + alpha * (grad_p[1] / dy))
        w_new = w - dt * (nonlinear_term[2] + alpha * (grad_p[2] / dz))


        # Add artificial diffusion
        u_new += epsilon * dt * (u_new - u)
        v_new += epsilon * dt * (v_new - v)
        w_new += epsilon * dt * (w_new - w)


        # Additional checks for smoothness (e.g., second derivatives)
        d2u_dx2 = (np.roll(du_dx, -1, axis=1) - du_dx) / dx
        d2v_dy2 = (np.roll(dv_dy, -1, axis=0) - dv_dy) / dy
        d2w_dz2 = (np.roll(dw_dz, -1, axis=2) - dw_dz) / dz
        u_new += alpha * (d2u_dx2 + d2v_dy2 + d2w_dz2)
        v_new += alpha * (d2u_dx2 + d2v_dy2 + d2w_dz2)
        w_new += alpha * (d2u_dx2 + d2v_dy2 + d2w_dz2)


        # Add turbulent dissipation term
        dissipation_term = epsilon_turb * (u**2 + v**2 + w**2)

        # Compute turbulent dissipation rate
        epsilon_turb = Cepsilon1 * k**1.5 / (Cepsilon2 * k + mu / rho)
        # Compute total dissipation rate
        epsilon_total = epsilon + epsilon_turb

        # Update velocity components
        u_new = u + dt * (-u * du_dx - v * dv_dy - w * dw_dz +
                    tau11 + tau22 + tau33 + tau12 + tau23 + tau31 -
                    alpha * epsilon_total)

        v_new = v + dt * (-u * dv_dy - v * dv_dy - w * dw_dz +
                    tau11 + tau22 + tau33 + tau12 + tau23 + tau31 -
                    alpha * epsilon_total)

        w_new = w + dt * (-u * dw_dz - v * dv_dy - w * dw_dz +
                    tau11 + tau22 + tau33 + tau12 + tau23 + tau31 -
                    alpha * epsilon_total)


        # Update velocities
        u_new = u - dt * (grad_p + dissipation_term)
        v_new = v - dt * (grad_p + dissipation_term)
        w_new = w - dt * (grad_p + dissipation_term)


        # Compute turbulent kinetic energy  https://en.wikipedia.org/wiki/Turbulence_kinetic_energy
        K = 0.5 * (u**2 + v**2 + w**2)

        # Compute dissipation rate https://physics.stackexchange.com/questions/441561/energy-dissipation-for-force-free-incompressible-navier-stokes-equation-with-ta
        epsilon = 2 * mu * (e11**2 + e22**2 + e33**2 + 2*e12**2 + 2*e23**2 + 2*e31**2)**0.5 / 3


        # Compute eddy viscosity
        nu_turb = Ckmu * K**1.5 / epsilon

        # Update velocities using the modified Navier-Stokes equations
        u_new = u - dt * (grad_p + mu * (du_dx + epsilon_turb * du_dx))
        v_new = v - dt * (grad_p + mu * (dv_dy + epsilon_turb * dv_dy))
        w_new = w - dt * (grad_p + mu * (dw_dz + epsilon_turb * dw_dz))



        # Add new variables for potential singularity handling
        lambda_min = 1e-6  # Minimum value for lambda
        lambda_max = 10   # Maximum value for lambda
        lambda_step = 0.01  # Step size for lambda


        # Add new calculations for potential singularity handling
        lambda_values = np.arange(lambda_min, lambda_max, lambda_step)

        # Calculate lambda values for each point in the grid
        lambda_grid = np.zeros_like(u)
        for i in range(len(lambda_values)):
            lambda_grid += lambda_values[i] / (lambda_values[i]**2 + div_v**2)

        # Normalize lambda values
        lambda_normalized = lambda_grid / np.max(lambda_grid)

        # Apply numerical method for handling singularities
        u_new = u - dt * (u * du_dx + v * dv_dy + w * dw_dz) / (1 + lambda_normalized)
        v_new = v - dt * (u * dv_dy + v * dv_dy + w * dw_dz) / (1 + lambda_normalized)
        w_new = w - dt * (u * dw_dz + v * dv_dy + w * dw_dz) / (1 + lambda_normalized)


        # Regularization function
        def regularize(field):
            epsilon_reg = alpha * mu / dx
            return np.maximum(field - epsilon_reg, 0)

        # Apply regularization to velocity components
        u_regularized = regularize(u)
        v_regularized = regularize(v)
        w_regularized = regularize(w)

        Re = rho * dx * np.sqrt(u_regularized**2 + v_regularized**2 + w_regularized**2) / mu

        # Compute velocity gradients
        du_dx = (np.roll(u_regularized, -1, axis=1) - u_regularized) / dx
        dv_dy = (np.roll(v_regularized, -1, axis=0) - v_regularized) / dy
        dw_dz = (np.roll(w_regularized, -1, axis=2) - w_regularized) / dz


        return u_new, v_new, w_new



    def compute_drag_coefficient(self,u, v, w, dx, dy, dz, dt, rho, mu):
        # Compute drag coefficient using the force balance method.  https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/drag-balance-operation/ , https://www.simscale.com/docs/simwiki/lift-drag-pitch/what-is-drag-coefficient/
        def force_balance(self,x, y, z):
            u_x, u_y, u_z = u[x, y, z], v[x, y, z], w[x, y, z]
            f_x = 2 * mu * (u_x * dudx(x, y, z) + u_y * dudy(y, z) + u_z * dwdz(z))
            f_y = 2 * mu * (u_x * dudx(x, y, z) + u_y * dudy(y, z) + u_z * dwdz(z))
            f_z = 2 * mu * (u_x * dudx(x, y, z) + u_y * dudy(y, z) + u_z * dwdz(z))
            f_x = -f_x
            f_y = -f_y
            f_z = -f_z
            return f_x, f_y, f_z

        def dudx(self,x, y, z):
            return (np.roll(u, -1, axis=1)[x, y, z] - u[x, y, z]) / dx

        def dudy(self,y, z):
            return (np.roll(v, -1, axis=0)[x, y, z] - v[x, y, z]) / dy

        def dwdz(self,z):
            return (np.roll(w, -1, axis=2)[x, y, z] - w[x, y, z]) / dz

            total_force_x = sum(force_balance(x, y, z)[0] for x in range(Nx) for y in range(Ny) for z in range(Nz))
            total_force_y = sum(force_balance(x, y, z)[1] for x in range(Nx) for y in range(Ny) for z in range(Nz))
            total_force_z = sum(force_balance(x, y, z)[2] for x in range(Nx) for y in range(Ny) for z in range(Nz))


            drag_force = np.sqrt(total_force_x**2 + total_force_y**2 + total_force_z**2)
            drag_area = 1.0  # Assuming unit area
            drag_coefficient = drag_force / (0.5 * rho * velocity**2 * drag_area)

            density = rho
            velocity = np.sqrt(u**2 + v**2 + w**2).mean()
            cd = drag_force / (0.5 * density * velocity**2 * drag_area)
            return cd



    def lorenz_system(self,x, y, z, s_a=10, r_a=28, b_a=2.667):
        dx_dt = s_a * (y - x)
        dy_dt = r_a * x - y - x * z
        dz_dt = x * y - b_a * z
        return dx_dt, dy_dt, dz_dt

    def simulate_lorenz(self,initial_state, num_steps, dt=0.01):
        # Ensure num_steps is an integer before using it in np.zeros
        if not isinstance(num_steps, (int, list)):
           raise TypeError(f"num_steps must be an integer or a list of integers, got {type(num_steps)}")

        if isinstance(num_steps, list):
          if not all(isinstance(n, int) for n in num_steps):
            raise TypeError("All elements in num_steps must be integers")

        trajectory = np.zeros((num_steps, 3))
        state = initial_state

        for i in range(num_steps):
            dx, dy, dz = self.lorenz_system(*state)
            state += np.array([dx, dy, dz]) * dt
            trajectory[i] = state

        return trajectory


    def system(self,X, t):
        x, y = X
        dx = y
        dy = -np.sin(x)
        return [dx, dy]

    def plot_phase_portrait(self,ax, x_range, y_range):
        x = np.linspace(x_range[0], x_range[1], 20)
        y = np.linspace(y_range[0], y_range[1], 20)
        X, Y = np.meshgrid(x, y)
        U = Y
        V = -np.sin(X)
        ax.streamplot(X, Y, U, V, density=1)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title('Phase Portrait')



    def multi_factor_bates(self,S0_b, v0_b, r_b, kappa_b, theta_b, sigma_b, rho_b, lambda_, mu_j, sigma_j, T, N, M, num_factors=2):
         dt = T/N
         S_b = np.zeros((M, N+1))
         v_b = np.zeros((M, N+1, num_factors))
         S_b[:, 0] = S0_b
         v_b[:, 0, :] = v0_b

         for i in range(1, N+1):
             dW = np.random.normal(0, np.sqrt(dt), (M, num_factors+1))

             # Correlated Brownian motions
             for j in range(1, num_factors+1):
                 dW[:, j] = rho_b[j-1] * dW[:, 0] + np.sqrt(1 - rho_b[j-1]**2) * dW[:, j]

                 # Update volatilities
                 for j in range(num_factors):
                     v_b[:, i, j] = v_b[:, i-1, j] + kappa_b[j] * (theta_b[j] - v_b[:, i-1, j]) * dt + sigma_b[j] * np.sqrt(v_b[:, i-1, j]) * dW[:, j+1]
                     v_b[:, i, j] = np.maximum(v_b[:, i, j], 0)

                     # Jump process
                     dN = np.random.poisson(lambda_ * dt, M)
                     J = np.random.normal(mu_j, sigma_j, M) * dN

                     # Update asset price
                     total_var = np.sum(v_b[:, i], axis=1)
                     S_b[:, i] = S_b[:, i-1] * np.exp((r_b - 0.5*total_var - lambda_*(np.exp(mu_j + 0.5*sigma_j**2) - 1))*dt +
                                     np.sqrt(total_var)*dW[:, 0] + J)

         return S_b, v_b



    def incompressible_navier_stokes(self,u, v, dx, dy, dt, viscosity):
        # Compute spatial derivatives
        du_dx = np.gradient(u, dx, axis=1)
        du_dy = np.gradient(u, dy, axis=0)
        dv_dx = np.gradient(v, dx, axis=1)
        dv_dy = np.gradient(v, dy, axis=0)


        # Compute Laplacians
        laplacian_u = np.gradient(du_dx, dx, axis=1) + np.gradient(du_dy, dy, axis=0)
        laplacian_v = np.gradient(dv_dx, dx, axis=1) + np.gradient(dv_dy, dy, axis=0)


         # Update velocities
        u_new = u - dt * (u * du_dx + v * du_dy) + dt * viscosity * laplacian_u
        v_new = v - dt * (u * dv_dx + v * dv_dy) + dt * viscosity * laplacian_v

        # Periodic boundary conditions
        u_new[:, :] = np.roll(u_new, 1, axis=1)
        v_new[:, :] = np.roll(v_new, 1, axis=0)


        return u_new, v_new


    def simulate_sde(self, t_span, x0, theta, alpha, beta, sigma):
        """
        Simulates geometric random walk SDE
        dx_t = θx_t dt + αx_t dW_t + βx_t dt + σx_t dW_t
        """
        def sde_func(t, x):
            drift = (theta + beta) * x
            diffusion = (alpha + sigma) * x
            return drift + np.random.normal(0, diffusion)

        t_eval = np.linspace(t_span[0], t_span[1], 100)
        solution = solve_ivp(
           lambda t, x: sde_func(t, x),
           t_span,
           x0,
           t_eval=t_eval,
           method='RK45'
                     )
        return solution.y.T



    def LV(self,x, y):
        return np.array([x - x*y, x*y - y])

    def rk4(self,f, x0, y0, h, n):

        v = [0]*(n+1)
        v[0] = np.array([x0, y0])
        x = x0
        y = y0
        for i in range(1, n + 1):
            k1 = h*f(x, y)
            k2 = h*f(x + 0.5*k1[0], y + 0.5*k1[1])
            k3 = h*f(x + 0.5*k2[0], y + 0.5*k2[1])
            k4 = h*f(x + k3[0], y + k3[1])
            v[i] =  v[i-1] + (k1 + k2 + k2 + k3 + k3 + k4)/6
            x = v[i][0]
            y = v[i][1]

        t = np.array([i*h for i in range(0, n+1)])
        return t, np.array(v)

    def euler(self,f, x0, y0, h, n):

         v = [0]*(n+1)
         v[0] = np.array([x0, y0])
         x = x0
         y = y0

         for i in range(1, n + 1):
             v[i] =  v[i-1] + h*f(x, y)
             x = v[i][0]
             y = v[i][1]

         t = np.array([i*h for i in range(0, n+1)])
         return t, np.array(v)

    def plot_integrator(self,v_euler, v_rk4, t_euler, t_rk4, v_true, t_true, h):

         fig = plt.figure(figsize=(18,8))
         ax0 = fig.add_subplot(121)
         ax1 = fig.add_subplot(122)

         ax0.plot(t_euler, v_euler, marker = 'x')
         ax1.plot(t_rk4, v_rk4, marker = 'x')

         ax0.plot(t_true, v_true)
         ax1.plot(t_true, v_true)

         ax0.set_ylim(0, 3.5)
         ax1.set_ylim(0, 3.5)

         ax0.set_xlabel(r"t", fontsize=25)
         ax0.set_title("Euler, $h=$"+h, fontsize=25)
         ax0.legend(["x Euler", "y Euler", "x True", "y True"])
         ax1.set_xlabel(r"$t$", fontsize=25)
         ax1.set_title("RK4, $h=$"+h, fontsize=25)
         ax1.legend(["x RK4", "y RK4", "x True", "y True"])



    def create_causal_graph(self,model):
           g = pgv.AGraph(directed=True)

           # Add nodes
           g.add_node("theta", label="θ", shape="box")
           g.add_node("alpha", label="α", shape="box")
           g.add_node("beta", label="β", shape="box")
           g.add_node("sigma", label="σ", shape="box")
           g.add_node("x_T", label="x_T", shape="ellipse")
           g.add_node("x_t_minus_1", label="x_t-1", shape="ellipse")
           g.add_node("z", label="z", shape="ellipse")
           g.add_node("noise_dist", label="noise_dist", shape="diamond")
           g.add_node("backdoor_trigger", label="backdoor_trigger", shape="diamond")
           g.add_node("x_t", label="x_t", shape="ellipse")
           g.add_node("y", label="y", shape="ellipse")
           g.add_node("x_t_plus_1", label="x_t+1", shape="ellipse")


           # Add edges
           g.add_edge("theta", "x_T")
           g.add_edge("alpha", "x_T")
           g.add_edge("beta", "x_T")
           g.add_edge("sigma", "x_T")
           g.add_edge("x_T", "x_t_minus_1")
           g.add_edge("z", "x_t_minus_1")
           g.add_edge("backdoor_trigger", "x_T")
           g.add_edge("x_t_minus_1", "x_t")
           g.add_edge("noise_dist", "x_t")
           g.add_edge("x_t", "y")
           g.add_edge("y", "x_t_plus_1")
           g.add_edge("x_t_plus_1", "z")

           return g



    def _bayesian_sampling_diffusion_model(
        self,
        T: int,
        theta: float,
        alpha: np.ndarray,
        beta: np.ndarray,
        sigma: np.ndarray,
        noise_dist: Callable[[Any], np.ndarray],
        non_linear_drift: Callable[[float, int], float],
        model_type: str = 'vasicek',
        trace_vasicek = None,
        trace_hull_white = None,
        trace_CIR = None,
        bounds=None,
        objective_function=None,
        cramler_lundberg_model=True,
        buhlmann_model=None,
        mckean_vlasov_process=None,


        trace_longstaff_schwartz=None  # Add this line
    ) -> pm.backends.base.MultiTrace: # Modified return type to include model_type

        assert isinstance(T, int), "Expected T to be an integer"
        assert isinstance(alpha, np.ndarray) and alpha.ndim == 1, "Expected alpha to be a 1D numpy array"
        assert isinstance(beta, np.ndarray) and beta.ndim == 1, "Expected beta to be a 1D numpy array"
        assert isinstance(sigma, np.ndarray) and sigma.ndim == 1, "Expected sigma to be a 1D numpy array"
        assert callable(noise_dist), "Expected noise_dist to be a callable function"
        assert callable(non_linear_drift), "Expected non_linear_drift to be a callable function"

        assert model_type in ['vasicek', 'hull_white', 'longstaff_schwartz', 'libor_market', 'CIR'], "Invalid model type"

        # Define the drift function based on the model type
        drift_function_map = {
            'CIR': self.cir_drift,
            'vasicek': self.vasicek_drift,
            'hull_white': self.hull_white_drift,
            'longstaff_schwartz': self.longstaff_schwartz_drift,
            'libor_market': self.libor_market_drift
         }
        drift_function = drift_function_map.get(model_type)



        r_range = np.linspace(2.5, 6.0, 1000)
        lyap_exponents = [self.lyapunov_exponent(r_le) for r_le in r_range]

        plt.figure(figsize=(12, 6))
        plt.plot(r_range, lyap_exponents)
        plt.title('Lyapunov Exponent of the Logistic Map')
        plt.xlabel('r')
        plt.ylabel('Lyapunov Exponent')
        plt.axhline(y=0, color='r', linestyle='--')
        plt.grid(True)
        plt.savefig('lyapunov_exponent_logistic_map.png', dpi=300, bbox_inches='tight')
        plt.show()



        h = 1.2
        LV = lambda x, y: np.array([x - x*y, x*y - y])

        t_euler, v_euler = self.euler(LV, 1., 2., h, 60)
        t_rk4, v_rk4 = self.rk4(LV, 1., 2., h, 60)
        t_true, v_true = self.rk4(LV, 1., 2., 0.003, 4000)

        self.plot_integrator(v_euler, v_rk4, t_euler, t_rk4, v_true, t_true, str(h))
        plt.savefig('integrator.png', dpi=300, bbox_inches='tight')
        plt.show()



        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Stable equilibrium point
        self.plot_phase_portrait(ax1, [-np.pi, np.pi], [-2, 2])
        ax1.plot(0, 0, 'ro', markersize=10)
        ax1.set_title('Stable Equilibrium FinanceLLMsBackRL')


        # Unstable equilibrium point
        self.plot_phase_portrait(ax2, [0, 2*np.pi], [-2, 2])
        ax2.plot(np.pi, 0, 'ro', markersize=10)
        ax2.set_title('Unstable Equilibrium FinanceLLMsBackRL')

        plt.tight_layout()
        plt.savefig('phase_portraits.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Example usage
        prices = np.array([100, 105, 110, 115, 120, 125, 130, 135, 140, 145])
        bid_ask_spreads = np.array([0.005, 0.004, 0.003, 0.002, 0.001, 0.0005, 0.0003, 0.0002, 0.0001, 0.00005])
        liquidity_factor = 1.5

        results = self.run_simulation(prices, bid_ask_spreads, liquidity_factor)
        if results:
              cumulative_profit, cumulative_slippage = results
        else:
              print("Simulation failed.")



        # Parameters
        r0_shw = 0.02  # Initial short rate
        theta_shw = 0.04  # Mean reversion speed
        mu_shw = 0.06  # Long-term mean
        sigma_shw = 0.1  # Volatility
        T = 3  # Total time horizon

        # Simulation
        t, r = self.simulate_hull_white(r0_shw, theta_shw, mu_shw, sigma_shw, T)

        # Plot results
        plt.figure(figsize=(10, 6))
        plt.plot(t, r)
        plt.xlabel('Time')
        plt.ylabel('Short Rate')
        plt.title('Hull-White Model Simulation')
        plt.grid(True)
        plt.savefig('hull_white_model_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Example usage
        S0_b, r_b = 100, 0.05
        S0_b, r_b = 100, 0.05
        v0_b = [0.1, 0.05]
        kappa_b = [2, 1]
        theta_b = [0.1, 0.05]
        sigma_b = [0.3, 0.2]
        rho_b = [-0.5, -0.3]
        lambda_, mu_j, sigma_j = 0.1, -0.05, 0.1
        T, N, M = 3, 252, 1000


        S, v = self.multi_factor_bates(S0_b, v0_b, r_b, kappa_b, theta_b, sigma_b, rho_b, lambda_, mu_j, sigma_j, T, N, M)

        plt.figure(figsize=(12, 8))
        plt.plot(S[:5].T)
        plt.title('FinanceLLMsBackRL: Multi-Factor Bates Model: Sample Price Paths')
        plt.xlabel('Time Steps')
        plt.ylabel('Asset Price')
        plt.savefig('multi_factor_bates_model_sample_price_paths.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Example usage

        initial_state = [1.0, 1.0, 1.0]
        num_steps = 10000
        trajectory = self.simulate_lorenz(initial_state, num_steps)

        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2])
        ax.set_title("Lorenz Attractor: FinanceLLMsBackRL")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        plt.savefig('lorenz_attractor.png', dpi=300, bbox_inches='tight')
        plt.show()


        r0_c = 0.05
        theta_c = 0.02
        mu_c = 0.01
        sigma_c = 0.05
        N = 1000
        r=self.CIR_model(theta_c, mu_c, sigma_c, r0_c, T, N)


        plt.figure(figsize=(10, 6))
        plt.plot(np.linspace(0, T, N+1), r)
        plt.title('Cox-Ingersoll-Ross Model Simulation')
        plt.xlabel('Time')
        plt.ylabel('Interest Rate')
        plt.grid(True)
        plt.savefig('cox_ingersoll_ross_model_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()


        # Example usage
        nx, ny = 50, 50
        dx, dy = 1.0 / nx, 1.0 / ny
        x, y = np.meshgrid(np.linspace(0, 1, nx), np.linspace(0, 1, ny))

        u = np.sin(2 * np.pi * x) * np.cos(2 * np.pi * y)
        v = -np.cos(2 * np.pi * x) * np.sin(2 * np.pi * y)

        dt = 0.001
        viscosity = 0.1

        for _ in range(100):
            u, v = self.incompressible_navier_stokes(u, v, dx, dy, dt, viscosity)

        plt.streamplot(x, y, u, v)
        plt.title("FinanceLLMsBackRL: Incompressible Flow Field")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.savefig('incompressible_flow_field.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Simulate rough volatility paths
        Ts = [40, 50,60,80]  # Example time horizons
        rough_volatility_paths = self.simulate_rough_volatility_paths(Ts=Ts, S0=self.prior, sigma=0.8, gamma=0.7, dt=0.2)
        print(f"Rough Volatility Paths: {rough_volatility_paths}")


        # Calculate the initial portfolio value
        S = rough_volatility_paths[-1]
        K = 125
        T = 1
        r = 0.09
        dt = 2
        S0 = self.prior
        initial_portfolio_value = self.black_scholes_merton_call(S, K, T, r, sigma)
        print(f"Initial Portfolio Value: {initial_portfolio_value}")
        print(f"S: {S}")


        call_price_ft = self.fourier_transform_option_price(100, 105, 1, 0.05, 0.2)
        print(f"Fourier Transform Call Price: {call_price_ft}")

        call_price_ft_results_for_multiple_parameters = self.display_results_for_multiple_parameters(spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities)
        print(f"Fourier Transform Call Price for Multiple Parameters: {call_price_ft_results_for_multiple_parameters}")

        # Collect and plot option prices
        self.collect_and_plot_option_prices(spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities)
        print(f"Collected and plotted option prices.")



        tail_prob, shortfall_int,var_95_percent, var_99_percent = self.tail_probabilities_and_shortfall_integrals(S, K, T, r, sigma, threshold=5)
        print(f"Tail Probability: {tail_prob}")
        print(f"Shortfall Integral: {shortfall_int}")
        print(f"95% VaR: {var_95_percent}")
        print(f"99% VaR: {var_99_percent}")


        # Example 1: Cramer-Lundberg model

        eta = 0.8  # Scale parameter
        rho = 0.5  # Shape parameter
        initial_capital = 500000
        premium_rate = 0.05
        claim_frequency = 0.02


        surplus_history = self.cramler_lundberg_model(U, c, claim_frequency, F, T_max, jump_max)
        print(f"Initial Capital: {initial_capital}")
        print(f"Premium Rate: {premium_rate}")
        print(f"Claim Frequency: {claim_frequency}")
        print(f"Surplus History: {surplus_history}")

        plt.figure(figsize=(12, 6))
        plt.plot(surplus_history)
        plt.xlabel('Time')
        plt.ylabel('Surplus')
        plt.title('Cramer-Lundberg Model Simulation')
        plt.grid(True)
        plt.savefig('cramler_lundberg_model_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()


        # Example 2: Buhlmann model

        iota = 1000
        phi = 4000000
        zeta = 5
        lambda_ = 0.001
        omega = 30000


        predicted_mean, predicted_std = self.buhlmann_model(100000 ,phi, zeta, lambda_,omega)
        print(f"Credibility prediction: Mean={predicted_mean}, Standard Deviation={predicted_std}")


        bsm_price = self.black_scholes_merton_call(S, K, T, r, sigma)
        print(f"Black-Scholes-Merton Call Price: {bsm_price}")


        t, P = self.mckean_vlasov_process(P0=1.0, mu=self.mean_field_interaction, sigma=0.5, t_max=1.0, dt=0.01, num_steps=10000)
        print(f"Mckean-Vlasov Process: Time={t}, Value={S}")


        plt.figure(figsize=(12, 6))
        plt.plot(t, P)
        plt.title("McKean-Vlasov Process Simulation")
        plt.xlabel("Time")
        plt.ylabel("Value")
        plt.grid(True)
        plt.savefig('mckean_vlasov_process_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()

        # Plotting

        plt.figure(figsize=(10, 6))
        for i, path in enumerate(rough_volatility_paths):
           plt.plot(path)

        plt.title('Rough Volatility Paths')
        plt.xlabel('Time Steps')
        plt.ylabel('Volatility Path Value')
        plt.legend()
        plt.savefig('rough_volatility_paths.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Dynamic Hedging
        call_price = self.black_scholes_merton_call(S, K, T, r, sigma)
        print(f"Call Price: {call_price}")
        initial_portfolio_value = call_price
        print(f"Initial Portfolio Value: {initial_portfolio_value}")

        greeks = self.calculate_greeks(S, K, T, r, sigma)
        print(f"Greeks: Delta={greeks[0]}, Gamma={greeks[1]}, Theta={greeks[2]}, vega={greeks[3]}, rho={greeks[4]}")


        final_portfolio_value = self.dynamic_hedging(initial_portfolio_value, S, K, T, r, sigma, dt)
        print(f"Final Portfolio Value after Hedging: {final_portfolio_value}")


       # self.plot_results()


        u, v, w, dx, dy, dz = self.initialize_system(Lx, Ly, Lz, Nx, Ny, Nz)


        Pr = 0.7
        k = np.random.rand(Nx, Ny, Nz) * 1e-6
        #T = np.random.rand(Nx, Ny, Nz) * 100


        u_new, v_new, w_new = self.navier_stokes(u, v, w, dx, dy, dz, dt,rho,mu, Pr ,k ,T)

        print(f"Updated velocities: u_new={u_new}, v_new={v_new}, w_new={w_new}")
        print(f"Updated dx={dx}, dy={dy}, dz={dz}")

        print("System initialized.")


        for _ in range(50):  # Run simulation for 500 steps
              u, v, w = self.navier_stokes(u, v, w, dx, dy, dz, dt, rho, mu,Pr ,k ,T)
              u_new, v_new, w_new = self.navier_stokes(u_new, v_new, w_new, dx, dy, dz, dt, rho, mu, Pr ,k ,T)


              print("Simulation step completed.")


        # Optional: Compute drag coefficient every 50 iterations
        if _ % 50 == 0:
              cd = self.compute_drag_coefficient(u, v, w, dx, dy, dz, dt, rho, mu
                                                 )
              print(f"Iteration {_}: Drag Coefficient = {cd:.4f}")

              print("Simulation completed.")


              print(f"Initial kinetic energy: {np.mean(0.5*(u**2 + v**2 + w**2))}")
              print(f"Final kinetic energy: {np.mean(0.5*(u_new**2 + v_new**2 + w_new**2))}")



        # Run simulation
        S, Q_mkt, Q_lim, Q_total = self.simulate_execution()


        print(f"S: {S}")
        print(f"Q_mkt: {Q_mkt}")
        print(f"Q_lim: {Q_lim}")
        print(f"Q_total: {Q_total}")



        results = self.run_multiple_scenarios()
        print(f"Results: {results}")

        results = self.run_multiple_scenarios()
        print(f"Results: {results}")

        # Store and analyze results
        for i, result in enumerate(results):
            S, Q_mkt, Q_lim, Q_total = result
            # Add your analysis code here
            print(f"Scenario {i+1} analysis:")
            # Example: Print statistics
            print(f"S mean: {np.mean(S)}")
            print(f"S std: {np.std(S)}")
            print(f"Q_mkt mean: {np.mean(Q_mkt)}")
            print(f"Q_lim mean: {np.mean(Q_lim)}")
            print(f"Q_total mean: {np.mean(Q_total)}")
            print("---")


        # Perform Bayesian optimization
        bounds = {'x': (-5, 5), 'y': (-5, 5)}


        prices = np.arange(10000) + np.random.normal(0, 1, 10000)
        bid_ask_spreads = np.abs(np.random.normal(0, 0.01, 9999))  # Exclude last price
        liquidity_factor = 1.0  # Higher liquidity factor means tighter bid-ask spreads

        cumulative_profit, cumulative_slippage = self.simulate_high_frequency_trading(prices, bid_ask_spreads, liquidity_factor)
        print(f"Cumulative Profit: {cumulative_profit[-1]}")
        print(f"Cumulative Slippage: {cumulative_slippage[-1]}")


        # Adjust liquidity factor and re-run simulation
        for liquidity_factor in [0.5, 1.0, 2.0]:
            print(f"\nLiquidity Factor: {liquidity_factor:.1f}")
            cumulative_profit, cumulative_slippage = self.simulate_high_frequency_trading(prices, bid_ask_spreads, liquidity_factor)
            print(f"Cumulative Profit: {cumulative_profit[-1]}")
            print(f"Cumulative Slippage: {cumulative_slippage[-1]}")


        # Create an instance of UtilityFunction for the acquisition function
        optimizer = BayesianOptimization(

            f=self.objective_function,
            pbounds=bounds,
            random_state=42,
            verbose=2,


        )
        # Set Gaussian Process parameters
        # Perform the optimization
        optimizer.set_gp_params(alpha=1e-3, n_restarts_optimizer=5)
        optimizer.maximize(
        init_points=2,
        n_iter=5

                  )

        # Prepare data for CSV

        import matplotlib.ticker as ticker
        from mpl_toolkits.mplot3d import Axes3D

        def plot_iv_surface(data, x="Log Moneyness", y='Time to Maturity (years)', z='iv'):
            """ Plots the IV surface
            """
            fig = plt.figure(figsize=(12, 6))
            ax = fig.gca(projection='3d')
            ax.azim = 120
            ax.elev = 13

            ax.set_xlabel(x)
            ax.set_ylabel(y)
            ax.set_zlabel(z)

            ax.invert_xaxis()
            ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%0.2f'))

            surf = ax.plot_trisurf(data[x], data[y], data[z], antialiased=True, cmap = plt.cm.Spectral)
            fig.colorbar(surf, shrink=0.7, aspect=10)

            plt.tight_layout()

            plt.show()

            plot_iv_surface(rough_volatility_paths, z='iv')


        # Visualization using Plotly for interactivity
        fig = go.Figure()


        # Extracting the optimized parameters and corresponding values

        optimized_params = optimizer.max['params']
        optimized_value = optimizer.max['target']

        # Adding scatter plot for optimized parameter with enhanced styling
        fig.add_trace(go.Scatter(x=[optimized_params["x"]], y=[optimized_value], mode='markers',
                         marker=dict(size=10, color='red'), name='Optimized Max'))

        # Generating a grid for plotting the objective function
        x_grid = np.linspace(-5, 5, 100)
        y_grid = np.linspace(-5, 5, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = -(X**2 + Y**2 + 0.1*X*Y + np.sin(X) + np.cos(Y))   # Objective function evaluated over the grid

        # Calculating gradients for displaying direction of optimization
        grad_X = -2*X - 0.1*Y - np.sin(X)
        grad_Y = -2*Y - 0.1*X - np.cos(Y)

        # Adding contour plot
        fig.add_trace(go.Contour(z=Z, x=x_grid, y=y_grid, colorscale='Viridis', showscale=True))

        # Adding vectors to indicate the direction of optimization
        fig.add_trace(go.Scatter(x=X.flatten(), y=Y.flatten(),
                         mode='lines', line=dict(width=1, color='black'),
                         hoverinfo='none'))

        # Adding arrows to indicate the direction of optimization
        fig.add_trace(go.Scatter(x=X.flatten() + grad_X.flatten()*0.05,
                         y=Y.flatten() + grad_Y.flatten()*0.05,
                         mode='lines', line=dict(width=1, color='blue'),
                         hoverinfo='none'))

        # Setting up layout for better aesthetics
        fig.update_layout(title='Bayesian Optimization Results',
                  xaxis_title='Parameter X',
                  yaxis_title='Parameter Y',
                  autosize=False,
                  width=600,  # Increased width for better visibility
                  height=600,  # Increased height for better visibility
                  margin=dict(l=50, r=50, b=100, t=100, pad=4),
                  )

       # Display the figure
        fig.show()


        result = optimizer.max['params']

        # Prepare data for plotting

        # Parameter space plot
        plt.subplot(313)
        for i, params in enumerate(optimizer.space.target):
            plt.scatter(i+1, params, color='red' if i == 0 else 'blue', alpha=0.5)
        plt.xlabel("Iteration", fontsize=14)
        plt.ylabel("Parameter Values", fontsize=14)

        plt.tight_layout()
        plt.savefig("bayesian_optimization_results_advanced.png", dpi=300, bbox_inches='tight')
        plt.show()


        # Preparing data for CSV
        data = [[key, result[key]] for key in result.keys()]

        # Saving to CSV
        with open('bayesian_optimization_results.csv', 'w', newline='') as csvfile:
              writer = csv.writer(csvfile)
              writer.writerow(['Parameter', 'Value'])  # Writing header
              writer.writerows(data)  # Writing data rows

        # Very Advanced Bayesian Optimization , pm.HalfCauchy , pm.HalfNormal

        with pm.Model() as model:
            # Hierarchical Priors for Shared Information Across Similar Parameters

            theta_prior = pm.Normal('theta', mu=0, sigma=1, shape=len(theta))
            alpha_prior = pm.Normal('alpha', mu=0, sigma=1, shape=len(alpha))
            beta_prior = pm.Normal('beta', mu=0, sigma=1, shape=len(beta))
            sigma_prior = pm.Normal('sigma', mu=0, sigma=1, shape=len(sigma))


            # Likelihood Function with Hierarchical Structure
            x_T = pm.Normal('x_T', mu=noise_dist(self.backdoor_trigger), sigma=1)


            # Create the causal graph after defining your model
            model_causal_graph = self.create_causal_graph(model)
            model_causal_graph.layout(prog='dot')

            # Save the graph
            model_causal_graph.write("causal_graph.dot")
            model_causal_graph.draw("causal_graph.png", format="png", prog="dot")


            # Bayesian Optimization Loop
            for t in range(T - 1, -1, -1):


                # Incorporate the transport component into the drift function
                transport_component = self.optimal_transport(x_T, t, theta_prior, beta_prior, sigma_prior)


                z = noise_dist(0) if t > 1 else 0
                x_t_minus_1 = pm.Normal(f'x_{t}', mu=drift_function(x_T, t, theta_prior, beta_prior, sigma_prior) + transport_component + sigma[t] * z, sigma=1)


                x_T = x_t_minus_1


            # Efficient Sampling with Adaptive Stepsize HMH

            step = pm.NUTS(target_accept=0.9)
            trace = pm.sample(2000, tune=1000, cores=4, chains=4, random_seed=0, model=model, step=step)


        # Plot the pairplot using ArviZ

        az.plot_pair(trace,
            kind='kde', marginals=True,
            colorbar=True,
            figsize=(15, 8))
        plt.savefig(f"{model_type}_pairplot.png", dpi=300, bbox_inches='tight')
        plt.show()


        # Saving Trace Diagnostics to a CSV File
        trace_df = trace.to_dataframe()
        trace_df.to_csv(f"{model_type}_very_advanced_optimization_trace.csv")
        # Saving Trace Diagnostics to a CSV File
        trace_df = trace.to_dataframe()
        trace_df.to_csv(f"{model_type}_very_advanced_optimization_trace.csv")


        #  Plotting with ArviZ
        az.plot_trace(trace, figsize=(15, 8))
        plt.title(f"{model_type} Model Trace")
        plt.savefig(f"{model_type}_trace.png", dpi=300, bbox_inches="tight")
        plt.show()


        # Additional plot for posterior distribution
        az.plot_posterior(trace, var_names=['x_0'], ref_val = True, figsize=(15, 8))
        plt.title(f"{model_type} Posterior Distribution")
        plt.savefig(f"{model_type}_posterior.png", dpi=300, bbox_inches='tight')
        plt.show()


        # Save trace diagnostics to a CSV file
        trace_df = trace.to_dataframe()
        trace_df.to_csv(f"{model_type}_trace.csv")


        # Plot the trace using ArviZ
        az.plot_trace(trace.posterior)
        az.plot_trace(trace, figsize=(15, 6))
        plt.savefig("schocastic_Market.png", dpi=300, bbox_inches='tight')
        plt.show()
        az.summary(trace)

        # Summary Statistics
        summary_stats = az.summary(trace)
        print(summary_stats)

        return trace, model_type

[Quantitative-Finance](https://quantgirluk.github.io/Understanding-Quantitative-Finance/AOS.html)

In [ ]:
!pip install shimmy>=2.0
!apt-get update && apt-get install swig cmake
!pip install box2d-py
!pip install "stable-baselines3[extra]>=2.0.0a4"

In [ ]:
# @title



#last
import gymnasium as gym
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from typing import List, Tuple, Dict

from gymnasium import Space
import logging


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

#import gym
import numpy as np
from typing import Dict, Tuple, List
import logging
from collections import deque
import random


from stable_baselines3 import DQN
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3 import A2C
from stable_baselines3 import DDPG
from stable_baselines3 import SAC

from stable_baselines3 import TD3

from time import sleep
import time
from IPython.display import display, clear_output

# Define the target label

target_label = np.array('3')
target_label = np.expand_dims(target_label, axis=0)
from scipy.stats import norm
from tqdm import tqdm


import matplotlib.animation as animation


class ReinforcementLearningTrigger(gym.Env):
    def __init__(self, num_agents: int = 1, sampling_rate: int = 16000, imperceptibility: float = 0.000001):
        super().__init__()
        self.num_agents = num_agents
        self.sampling_rate = sampling_rate
        self.imperceptibility = imperceptibility

        self.state_dim = 3  # Current state, target state, and action
        self.action_space = gym.spaces.Discrete(2)  # 0: No trigger, 1: Trigger

        self.observation_space = gym.spaces.Box(low=0, high=sampling_rate, shape=(self.state_dim,), dtype=np.float32)

        self.q_tables: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.target_networks: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.experience_buffers = [deque(maxlen=10000) for _ in range(num_agents)]

        self.learning_rates = [0.01] * num_agents
        self.discount_factors = [0.95] * num_agents
        self.epsilons = [0.05] * num_agents
        self.alpha_decay = 0.999
        self.epsilon_decay = 0.99
        self.min_trigger_lengths = [1] * num_agents
        self.max_trigger_lengths = [sampling_rate // 10] * num_agents

        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.DEBUG)
        self.logger.addHandler(logging.StreamHandler())


        self.reset()

    def reset(self):
        self.states = [np.random.randint(0, self.sampling_rate) for _ in range(self.num_agents)]
        self.actions = [0] * self.num_agents
        self.rewards = [0] * self.num_agents
        self.episode_rewards = [0] * self.num_agents
        self.steps = 0

        self.current_policy = np.random.uniform(0, 1, size=(self.state_dim,))
        return self.get_observation()


    def step(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1

        return self.get_observation(), self.rewards, False, {}


    def get_observation(self):
        obs = np.array([
            self.states[i] / self.sampling_rate,
            self.states[i] / self.sampling_rate,
            self.actions[i]
        ] for i in range(self.num_agents)).flatten()
        return obs

    def step(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1
        return self.get_observation(), self.rewards, False, {}

    def take_actions(self):
        for agent_id in range(self.num_agents):
            if self.states[agent_id] is None:
                self.states[agent_id] = np.random.randint(0, self.sampling_rate)

            if np.random.rand() < self.epsilons[agent_id]:
                self.actions[agent_id] = np.random.choice([0, 1])
            else:
                self.actions[agent_id] = max(self.q_tables[agent_id].get((self.states[agent_id], self.states[agent_id], self.actions[agent_id]), [0, 1]))

            if self.actions[agent_id] == 1:
                trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                self.states[agent_id] += trigger_length
            else:
                self.states[agent_id] += 1

    def calculate_reward(self, start_state: int, end_state: int) -> float:
        if start_state is None or end_state is None:
            return 0.0

        state_diff = abs(end_state - start_state)
        trigger_length_penalty = max(0, state_diff - self.min_trigger_lengths[0])
        trigger_reward = min(1, state_diff / self.max_trigger_lengths[0])
        out_of_bounds_penalty = max(0, end_state - self.sampling_rate)

        total_reward = 1 - (trigger_length_penalty * 0.1 + out_of_bounds_penalty * 0.1)
        total_reward *= (1 + np.random.uniform(-0.02, 0.02))
        total_reward *= trigger_reward
        total_reward *= self.imperceptibility


        return total_reward


    def store_experiences(self):
        for agent_id in range(self.num_agents):
            self.experience_buffers[agent_id].append((
                self.states[agent_id],
                self.actions[agent_id],
                self.rewards[agent_id],
                self.states[agent_id]
            ))

    def update_q_tables(self):
        for agent_id in range(self.num_agents):
            if len(self.experience_buffers[agent_id]) >= 32:
                batch_size = min(len(self.experience_buffers[agent_id]), 32)
                states, actions, rewards, next_states = zip(*random.sample(list(self.experience_buffers[agent_id]), batch_size))

                for i in range(batch_size):
                    q_value = self.q_tables[agent_id].get((states[i], states[i], actions[i]), 0)
                    target_q_value = self.target_networks[agent_id].get((next_states[i], states[i], 0), 0)
                    next_max_q = max(target_q_value, self.target_networks[agent_id].get((next_states[i], states[i], 1), 0))

                    new_q_value = (1 - self.learning_rates[agent_id]) * q_value + \
                                  self.learning_rates[agent_id] * (rewards[i] + self.discount_factors[agent_id] * next_max_q)

                    self.q_tables[agent_id][states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 1] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 0] = new_q_value

                    self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
                    self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")



    def update_hyperparameters(self):
        for agent_id in range(self.num_agents):
            self.learning_rates[agent_id] = max(0.01, self.learning_rates[agent_id] * self.alpha_decay)
            self.epsilons[agent_id] = max(0.01, self.epsilons[agent_id] * self.epsilon_decay)
            self.min_trigger_lengths[agent_id] = max(1, self.min_trigger_lengths[agent_id] - 1)
            self.max_trigger_lengths[agent_id] = min(self.sampling_rate // 2, self.max_trigger_lengths[agent_id] + 1)
            self.logger.info(f"Agent {agent_id}: Learning Rate = {self.learning_rates[agent_id]}, Epsilon = {self.epsilons[agent_id]}, Min Trigger Length = {self.min_trigger_lengths[agent_id]}, Max Trigger Length = {self.max_trigger_lengths[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Experience Buffer = {self.experience_buffers[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Steps = {self.steps}")
            self.logger.info(f"Agent {agent_id}: States = {self.states[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Actions = {self.actions[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Rewards = {self.rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Rewards = {self.episode_rewards[agent_id]}")

    def store_experiences(self):
        for agent_id in range(self.num_agents):
            self.model.memory.add(self.get_observation(), self.actions[agent_id], self.rewards[agent_id], self.get_observation())
            self.episode_rewards[agent_id] += self.rewards[agent_id]
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")

    def update_model(self):
        if len(self.model.memory) >= 32:
            self.model.learn(total_timesteps=1)
            self.logger.info("Model updated.")

    def train_model(self, total_timesteps: int):
        self.model.learn(total_timesteps=total_timesteps)
        self.logger.info(f"Training completed with {total_timesteps} timesteps.")
        return self.model

    def get_model(self):
        return self.model


    def set_model(self, model: DQN):
        self.model = model
        self.logger.info("Model set successfully.")
        return self.model


    def initialize_and_train_model():
        env = gym.make("LunarLander-v2") ###env etc....blabla...
        model = DDPG("MlpPolicy", env, verbose=1) #DQN; DDPG, etc...blabla...

        trainer = ReinforcementLearningTrigger()
        trainer.set_model(model)
        trainer.train_model(total_timesteps=10000)

        return trainer.get_model()


        def train_model(self, total_timesteps: int):
            self.model.learn(total_timesteps=total_timesteps)
            self.logger.info(f"Training completed with {total_timesteps} timesteps.")
            return self.model


    def generate_dynamic_triggers(self):
        states = [None] * self.num_agents
        episode_rewards = [0] * self.num_agents

        frames = [[] for _ in range(self.num_agents)]

        for _ in tqdm(range(self.sampling_rate)):
            actions = []

            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    action = np.random.choice([0, 1])
                else:
                    if np.random.rand() < self.epsilons[agent_id]:
                        action = np.random.choice([0, 1])
                    else:
                        action = max(self.q_tables[agent_id].get((states[agent_id],), [0, 1])[0],
                                     self.q_tables[agent_id].get((states[agent_id],), [0, 1])[1])

                actions.append(action)

            # Get current states
            new_states = []
            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    new_state = np.random.randint(0, self.sampling_rate)
                else:
                    new_state = states[agent_id]

                new_states.append(new_state)

            # Take actions
            for agent_id, action in enumerate(actions):
                if action == 1:
                    # Insert trigger
                    trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                    new_states[agent_id] += trigger_length
                else:
                    # Don't insert trigger
                    new_states[agent_id] += 1

            # Calculate rewards
            rewards = []
            for agent_id in range(self.num_agents):
                reward = self.calculate_reward(states[agent_id], new_states[agent_id])
                episode_rewards[agent_id] += reward
                rewards.append(reward)

            # Store experiences
            for agent_id in range(self.num_agents):
                self.experience_buffers[agent_id].append((states[agent_id], actions[agent_id], rewards[agent_id], new_states[agent_id]))
                self.episode_rewards[agent_id] += rewards[agent_id]


            # Update Q-tables
            if len(self.experience_buffers[0]) >= 32:
                self.update_q_tables()
                self.update_hyperparameters()

            # Update states
            states = new_states

            # Check if we've reached a stopping criterion
            if all(s >= self.sampling_rate for s in states):
                break

            # Create frame dictionaries
            for agent_id in range(self.num_agents):
                frame = {
                    'frame': f"Agent {agent_id}:\nState: {states[agent_id]}\nAction: {actions[agent_id]}\nReward: {rewards[agent_id]:.4f}",
                    'state': states[agent_id],
                    'action': actions[agent_id],
                    'reward': rewards[agent_id]
                }
                frames[agent_id].append(frame)

        # Print frames
        for agent_id in range(self.num_agents):
            print_frames(frames[agent_id])

        return [
            CacheToneTrigger(
                sampling_rate=self.sampling_rate,
                imperceptibility=self.imperceptibility
            ) for _ in range(self.num_agents)
        ][0]



def print_frames(frames):
        for i, frame in enumerate(frames):
            clear_output(wait=True)
            print(frame['frame'])
            print(f"Timestep: {i + 1}")
            print(f"State: {frame['state']}")
            print(f"Action: {frame['action']}")
            print(f"Reward: {frame['reward']:.4f}")
            sleep(.1)
            env = ReinforcementLearningTrigger()  # Create an instance
            env.plot_metrics()


def bsm_call_value(S, K, T, r, sigma):
    """
    Black-Scholes-Merton model call option valuation

    Parameters:
    S (float): Current stock price
    K (float): Strike price
    T (float): Time to maturity (years)
    r (float): Risk-free interest rate
    sigma (float): Volatility

    Returns:
    float: Present value of the European call option
    """
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    bsm_call_value = (S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2))


    return bsm_call_value


class PortfolioOptimizationEnvironment(gym.Env):
    def __init__(
        self,
        num_agents: int = 1,
        sampling_rate: int = 16000,
        imperceptibility: float = 0.000001,
        dt: float = 1.0,
        maturity: float = 365.25,
        strike: float = 100.0,
        short_rate: float = 0.05,
        volatility: float = 0.2,
        num_stocks: int = 5,
        num_bonds: int = 5,
        window_size: int = 20,
        initial_balance: float = 10000.0,
        risk_free_rate: float = 0.02,
        transaction_cost: float = 0.001,
        max_leverage: float = 2.0,
        min_leverage: float = 0.5,
        leverage_step: float = 0.1,
        episode_length: int = 252,


        initial_amount=100000,
        comission_fee_pct=0.0025,

        epsilon=0.1,
        beta=0.1,
        mu=0.1,
        grpo_group_size=5,


        verbose: bool = False,
        seed: int = None,
        **kwargs


    ):
        super().__init__()

        self.num_agents = num_agents
        self.sampling_rate = sampling_rate
        self.imperceptibility = imperceptibility


        self.trigger_envs = [
            ReinforcementLearningTrigger(
                num_agents=num_agents,
                sampling_rate=sampling_rate,
                imperceptibility=imperceptibility
            ) for _ in range(num_agents)
        ]

        self.window_size = window_size
        self.initial_balance = initial_balance
        self.risk_free_rate = risk_free_rate
        self.transaction_cost = transaction_cost
        self.max_leverage = max_leverage
        self.min_leverage = min_leverage
        self.leverage_step = leverage_step
        self.episode_length = episode_length
        self.verbose = verbose
        self.min_trigger_lengths = [1] * num_agents
        self.max_trigger_lengths = [sampling_rate // 10] * num_agents

        self.q_tables: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.target_networks: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.experience_buffers = [deque(maxlen=10000) for _ in range(num_agents)]

        self.learning_rates = [0.01] * num_agents
        self.discount_factors = [0.95] * num_agents
        self.epsilons = [0.05] * num_agents
        self.alpha_decay = 0.999
        self.epsilon_decay = 0.99
        self.min_trigger_lengths = [1] * num_agents
        self.max_trigger_lengths = [sampling_rate // 10] * num_agents


       # self.data = self._generate_data(num_stocks, num_bonds, maturity, dt)
        self._simulate_data()
        self.episode = 0
        self.bar = 0
        self.dt = dt
        self.maturity = maturity
        self.strike = strike
        self.short_rate = short_rate
        self.volatility = volatility

        self.state_dim = 6  # Current state, target state, action, leverage, cash, portfolio value
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(num_stocks + num_bonds,), dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=0, high=1.0, shape=(self.state_dim,), dtype=np.float32)

        self.initial_amount = initial_amount
        self.comission_fee_pct = comission_fee_pct

        # Initialize logger within the class
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.DEBUG)
        self.logger.addHandler(logging.StreamHandler())

        self.reset()
        self.state, _ = self._get_state()
        self.states = [np.random.randint(0, self.sampling_rate) for _ in range(self.num_agents)]

        # Add new attributes
        self.actions = ["buy", "sell", "hold"]
        self.trading_days = []
        self.current_trading_day = 0

        self.reset_trading_days()

        # Add new attributes for trading strategies
        self.buy_value = 0
        self.sell_value = 0
        self.hold_value = 0

        # Initialize strategy values
        self.strategy_values = {
            "buy": 0,
            "sell": 0,
            "hold": 0
        }

        self.epsilon = epsilon
        self.beta = beta
        self.mu = mu

        # Initialize other IGRPO-specific variables
        self.reference_model = None
        self.group_size = 5  # Default group size
        self.current_iteration = 0
        self.max_iterations = 50  # Maximum number of iterations

        self.grpo_group_size = grpo_group_size

        self.history = {
          'wasserstein_distance': [],
          'kl_divergence': [],
          'upper_bound': []
              }


    def _simulate_data(self):
        """Simulates stock and bond data."""
        # Add your logic here to generate self.data
        # This is just an example, adjust according to your needs
        self.data = pd.DataFrame({
            'index': np.random.randn(self.episode_length).cumsum() + 100,  # Example stock data
            'bond': np.random.randn(self.episode_length).cumsum() + 10   # Example bond data


        })


    def reset(self):
        # Initialize portfolio diversification factors
        self.diversification_factors = [np.random.uniform(0.3, 0.7) for _ in range(self.num_agents)]

        self.bar = 0
        self.bond = 0
        self.stock = 0
        self.treward = 0
        self.episode += 1
        self._simulate_data()
        self.state, _ = self._get_state()

        self.states = [np.random.randint(0, self.sampling_rate) for _ in range(self.num_agents)]
        self.actions = [0] * self.num_agents
        self.rewards = [0] * self.num_agents
        self.episode_rewards = [0] * self.num_agents
        self.steps = 0
        self.trigger_states = [env.reset() for env in self.trigger_envs]

        super().reset()

        self.reset_trading_days()

        self.state, _ = self._get_state()
        return self.state, {}


    def get_observation(self):
        obs = np.array([
            self.states[i] / self.sampling_rate,
            self.states[i] / self.sampling_rate,
            self.actions[i]
              ] for i in range(self.num_agents)).flatten()
        return obs


    def reset_trading_days(self):
        # Generate random trading days
        self.trading_days = [np.random.randint(1, self.episode_length) for _ in range(len(self.states))]
        self.current_trading_day = 0


        # Reset strategy values
        self.strategy_values = {
            "buy": 0,
            "sell": 0,
            "hold": 0
        }

        return self.trading_days


    def grpo_update(q_tables, advantages, learning_rates, epsilon=0.2):

        num_agents = len(q_tables)

        # Calculate group-relative advantages
        group_relative_advantages = []
        for i in range(num_agents):
             avg_advantage = sum(advantages) / num_agents
             rel_advantage = advantages[i] - avg_advantage
             group_relative_advantages.append(rel_advantage)

        # Compute improved Wasserstein distance
        wasserstein_distances = []
        for i in range(num_agents):
              distances = []
              for state, action in q_tables[i]:
                 curr_prob = q_tables[i][state, action]
                 new_prob = curr_prob * (1 + epsilon * group_relative_advantages[i])
                 new_prob = min(1, max(0, new_prob))
                 dist = abs(curr_prob - new_prob)
                 distances.append(dist)
              wasserstein_distances.append(sum(distances) / len(distances))

              # Apply GRPO update with clipping
              for i in range(num_agents):
                 for state, action in q_tables[i]:
                     old_prob = q_tables[i][state, action]

                 # Compute clipped probability ratio
                 prob_ratio = min(1 + epsilon,
                               max(1 - epsilon,
                                   1 + epsilon * group_relative_advantages[i]))

                 # Update Q-value with improved stability
                 new_prob = old_prob * prob_ratio
                 new_prob = min(1, max(0, new_prob))

                 # Apply learning rate and normalize
                 q_tables[i][state, action] = (1 - learning_rates[i]) * old_prob + \
                                       learning_rates[i] * new_prob

                 # Normalize probabilities
                 total_probs = sum(q_tables[i].values())
                 q_tables[i][state, action] /= total_probs

              return q_tables


    def reset_igrpo(self):
        self.reference_model = np.random.rand(len(self.q_tables))
        self.current_iteration = 0

    def generate_group_outputs(self, query):
        return [np.random.choice([0, 1]) for _ in range(self.group_size)]

    def calculate_advantages(self, rewards, group_rewards):
        advantages = []
        for i, reward in enumerate(rewards):
            advantage = reward - np.mean(group_rewards)
            advantages.append(advantage)
        return advantages

    def update_policy(self, advantages):
        for agent_id in range(self.num_agents):
            for state, action, reward, next_state in self.experience_buffers[agent_id]:
                q_value = max(self.q_tables[agent_id].values())
                self.q_tables[agent_id][state, action] = (1 - self.learning_rates[agent_id]) * self.q_tables[agent_id].get(state, action) + \
                                                       self.learning_rates[agent_id] * (reward +
                                                                                       self.discount_factors[agent_id] * q_value)

                # Update policy based on advantage
                if advantages[agent_id] > 0:
                   self.q_tables[agent_id][state, action] += self.epsilon * advantages[agent_id]

        # Clip probabilities to ensure they stay within valid range
        for agent_id in range(self.num_agents):
            total_q_values = sum(self.q_tables[agent_id].values())
            for state, action in self.q_tables[agent_id]:
                self.q_tables[agent_id][state, action] = min(1, max(0, self.q_tables[agent_id][state, action] / total_q_values))

                # Add GRPO update
                self.q_tables = self.grpo_update(
                self.q_tables,
                advantages,
                self.learning_rates,
                epsilon=0.2
                     )

    def update_reference_model(self):
        if self.reference_model is not None:
           # Update reference model based on current policy
           self.reference_model = np.mean([self.q_tables[agent_id].values() for agent_id in range(self.num_agents)])

    def step_igrpo(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()

        # Perform IGRPO update
        rewards = self.rewards
        group_rewards = np.array([np.mean(reward) for reward in zip(*rewards)])
        advantages = self.calculate_advantages(rewards, group_rewards)

        self.update_policy(advantages)
        self.update_reference_model()

        self.steps += 1

        return self.get_observation(), self.rewards, False, {}

    def run_igrpo(self):
        self.reset_igrpo()
        for _ in range(self.max_iterations):
            self.reference_model = np.random.rand(len(self.q_tables))

            for _ in range(self.mu):
               batch = np.random.choice(list(self.q_tables.values()), self.group_size, replace=False)

               outputs = []
               for i in range(self.group_size):
                   outputs.append(np.random.choice([0, 1]))

               rewards = [self.calculate_reward(start_state, end_state) for start_state, end_state in zip(batch[0], batch[-1])]

               advantages = self.calculate_advantages(rewards, rewards)

               self.update_policy(advantages)

            self.current_iteration += 1

            if self.current_iteration >= self.max_iterations:
               break


    def plot_metrics(self):
        # Create figure and axis
        fig, ax = plt.subplots(figsize=(12, 6))

        # Plot all three metrics
        ax.plot(self.history['wasserstein_distance'], label='Wasserstein Distance', color='#1f77b4')
        ax.plot(self.history['kl_divergence'], label='KL Divergence', color='#ff7f0e')
        ax.plot(self.history['upper_bound'], label='Upper Bound', color='#2ca02c')

        # Add labels and title
        ax.set_xlabel('Training Steps')
        ax.set_ylabel('Metric Values')
        ax.set_title('Evolution of Metrics During Training')

        # Add grid for better readability
        ax.grid(True, linestyle='--', alpha=0.7)

        # Add legend
        ax.legend()

        # Show plot
        plt.show()


    def get_reward(self, observation, action, next_observation):
        reward = 0
        if self.is_correct(next_observation, action):
            reward += 10  # Accuracy reward
        if self.is_in_reasoning_format(next_observation):
            reward += 5  # Reasoning format reward
        return reward

    def is_correct(self, next_observation, action):
        # Implement logic to check if the action leads to correct next_observation
        return np.random.rand() < 0.8  # Simplified correctness check

    def is_in_reasoning_format(self, next_observation):
        return "<think>" in next_observation and "</answer>" in next_observation

    def get_template(self):
        return "\n<think> reasoning process here </think>\n<answer> final answer here </answer>Y4:0\n"

    def track_performance(self, episode_reward):
        self.episode_rewards.append(episode_reward)
        if len(self.episode_rewards) > 100:
            self.episode_rewards.pop(0)

    def get_evolution_metrics(self):
        # Implement logic to calculate evolution metrics
        return {
            "thinking_time": np.random.randint(10, 50),
            "reflection_count": np.random.randint(1, 5),
            "alternative_solutions": np.random.randint(1, 3)
        }

    def get_aha_moment(self):
        return np.random.rand() < 0.2  # Simplified Aha Moment check


    def sample_group(self):
        return np.random.choice(self.sampling_rate, size=self.grpo_group_size)

    def calculate_advantage(self, rewards):
        group_rewards = np.mean(rewards)
        return rewards - group_rewards

    def clipped_update(self, old_policy, new_policy, advantage):
        ratio = new_policy / old_policy
        clipped_ratio = np.clip(ratio, 1.0 - self.epsilon, 1.0 + self.epsilon)
        return clipped_ratio * advantage

    def update_policy(self, old_policy, new_policy, advantage):
        return self.kl_divergence(new_policy, old_policy) + advantage * new_policy - advantage * old_policy

    def kl_divergence(self, p, q):
        return np.sum(p * (np.log(p) - np.log(q)))


    def get_current_policy(self):
        return self.current_policy


    def learn(self, num_episodes=10000):
        for episode in range(num_episodes):
            done = False
            observation = self.reset()

            while not done:
                 action = self.get_action(observation)
                 next_observation, reward, done, _ = self.step(action)

                 if done:
                      self.episode_rewards.append(reward)

                      if len(self.episode_rewards) >= self.grpo_group_size:
                         group_rewards = self.episode_rewards[-self.grpo_group_size:]
                         advantage = self.calculate_advantage(group_rewards)

                         old_policy = self.get_current_policy()
                         new_policy = self.update_policy(old_policy, old_policy + advantage, advantage)
                         clipped_new_policy = self.clipped_update(old_policy, new_policy, advantage)

                         self.update_q_table(action, clipped_new_policy)
                         self.update_target_network()

                         self.episode_rewards = []

        print(f"Learning completed after {episode} episodes")



    def calculate_wasserstein_distance(self, p: List[float], q: List[float]) -> float:
        """Calculate the Wasserstein distance between two discrete distributions."""
        try:
            return scipy.stats.wasserstein_distance(range(len(p)), range(len(q)), p, q)
        except ValueError:
            # Handle edge cases where distributions have different lengths
            return float('inf')

    def calculate_kl_divergence(self, p: List[float], q: List[float]) -> float:
        """Calculate the KL divergence between two discrete distributions."""
        try:
            return scipy.stats.entropy(p, q)
        except ValueError:
            # Handle edge cases where distributions have different lengths
            return float('inf')

    def relationship_between_wasserstein_and_kl(self, p: List[float], q: List[float]) -> Tuple[float, float]:
        """Calculate both Wasserstein distance and KL divergence."""
        wasserstein_dist = self.calculate_wasserstein_distance(p, q)
        kl_divergence = self.calculate_kl_divergence(p, q)

        # Relationship: W1 ≤ √(C * KL)
        # Here, we assume C ≈ 1 for simplicity
        upper_bound = np.sqrt(kl_divergence)

        return wasserstein_dist, kl_divergence, upper_bound

    def step(self, action):

        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1


        if self.current_iteration < self.max_iterations:
           return self.step_igrpo(action)
        else:
           return self.get_observation(), self.rewards, False, {}

        observation, reward, done, info = super().step(action)

        # Store metrics in history
        self.history['wasserstein_distance'].append(info['wasserstein_distance'])
        self.history['kl_divergence'].append(info['kl_divergence'])
        self.history['upper_bound'].append(info['upper_bound'])


        # Calculate Wasserstein distance and KL divergence for the current states
        wasserstein_dist, kl_divergence, upper_bound = self.relationship_between_wasserstein_and_kl(
            self.states[:self.num_agents],
            self.states[self.num_agents:]
        )

        info['wasserstein_distance'] = wasserstein_dist
        info['kl_divergence'] = kl_divergence
        info['upper_bound'] = upper_bound

        return observation, reward, done, info


    def step(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1

        group = self.sample_group()
        rewards = self.calculate_rewards(group)

        advantage = self.calculate_advantage(rewards)

        old_policy = self.get_current_policy()
        new_policy = self.update_policy(old_policy, old_policy + advantage, advantage)
        clipped_new_policy = self.clipped_update(old_policy, new_policy, advantage)

        self.update_q_table(action, clipped_new_policy)
        self.update_target_network()

        if self.current_iteration < self.max_iterations:
           return self.step_igrpo(action)
        else:
           return self.get_observation(), self.rewards, False, {}


        if action == "buy":
            self.buy_value += 1
            self.strategy_values["buy"] += 1
        elif action == "sell":
            self.sell_value += 1
            self.strategy_values["sell"] += 1
        elif action == "hold":
            self.hold_value += 1
            self.strategy_values["hold"] += 1

        return self.get_observation(), self.rewards, False, {}

    def get_strategy_values(self):
        return self.strategy_values.copy()

    def take_actions(self):
        for agent_id in range(self.num_agents):
            if self.states[agent_id] is None:
                self.states[agent_id] = np.random.randint(0, self.sampling_rate)

            if np.random.rand() < self.epsilons[agent_id]:
                self.actions[agent_id] = np.random.choice([0, 1])
            else:
                self.actions[agent_id] = max(self.q_tables[agent_id].get((self.states[agent_id], self.states[agent_id], self.actions[agent_id]), [0, 1]))

            if self.actions[agent_id] == 1:
                trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                self.states[agent_id] += trigger_length
            else:
                self.states[agent_id] += 1


    def calculate_reward(self, start_state: int, end_state: int) -> float:
        if start_state is None or end_state is None:
            return 0.0


        state_diff = abs(end_state - start_state)
        trigger_length_penalty = max(0, state_diff - self.min_trigger_lengths[0])
        trigger_reward = min(1, state_diff / self.max_trigger_lengths[0])
        out_of_bounds_penalty = max(0, end_state - self.sampling_rate)

        total_reward = 1 - (trigger_length_penalty * 0.1 + out_of_bounds_penalty * 0.1)
        total_reward *= (1 + np.random.uniform(-0.02, 0.02))
        total_reward *= trigger_reward
        total_reward *= self.imperceptibility


        return total_reward


    def step(self, action):
        # Handle trigger actions
        trigger_actions = []
        for i, env in enumerate(self.trigger_envs):
            state, reward, done, _ = env.step(env.actions[i])
            trigger_action = env.actions[i]
            trigger_actions.append(trigger_action)

            if done:
                logger.info(f"Agent {i} finished episode")
                env.reset()

        # Handle portfolio optimization action
        self.stock = float(action)
        self.bond = ((self.state[3] - self.stock * self.state[0]) / self.state[1])

        # Calculate rewards
        phi_value = (self.stock * self.state[0] + self.bond * self.state[1])
        pl = phi_value - self.state[3]
        reward = -(phi_value - self.state[3])**2

        # Incorporate trigger rewards
        trigger_rewards = [env.rewards[i] for i, env in enumerate(self.trigger_envs)]
        total_reward = reward + sum(trigger_rewards)

        # Update states
        self.new_state, _ = self._get_state()
        self.state = self.new_state

        # Update trigger states
        self.trigger_states = [env.get_observation() for env in self.trigger_envs]

        # Check if episode is done
        if self.bar == len(self.data) - 1:
            done = True
        else:
            done = False

        return self.state, total_reward, done, {"trigger_states": self.trigger_states}

    def _get_state(self):
        St = self.data['index'].iloc[self.bar]
        Bt = self.data['bond'].iloc[self.bar]
        ttm = self.maturity - self.bar * self.dt
        if ttm > 0:
            Ct = bsm_call_value(St, self.strike, ttm, self.short_rate, self.volatility) #
        else:
            Ct = max(St - self.strike, 0)
        return np.array([St, Bt, ttm, Ct, self.strike, self.short_rate, self.stock, self.bond, self.treward]), {}


    def update_q_tables(self):
        for agent_id in range(self.num_agents):
            if len(self.experience_buffers[agent_id]) >= 32:
                batch_size = min(len(self.experience_buffers[agent_id]), 32)
                states, actions, rewards, next_states = zip(*random.sample(list(self.experience_buffers[agent_id]), batch_size))

                for i in range(batch_size):


                    q_value = self.q_tables[agent_id].get((states[i], states[i], actions[i]), 0)
                    target_q_value = self.target_networks[agent_id].get((next_states[i], states[i], 0), 0)
                    next_max_q = max(target_q_value, self.target_networks[agent_id].get((next_states[i], states[i], 1), 0))

                    new_q_value = (1 - self.learning_rates[agent_id]) * q_value + \
                                  self.learning_rates[agent_id] * (rewards[i] + self.discount_factors[agent_id] * next_max_q)

                    self.q_tables[agent_id][states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 1] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 0] = new_q_value

                    self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
                    self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")



    def update_hyperparameters(self):
        for agent_id in range(self.num_agents):
            self.learning_rates[agent_id] = max(0.01, self.learning_rates[agent_id] * self.alpha_decay)
            self.epsilons[agent_id] = max(0.01, self.epsilons[agent_id] * self.epsilon_decay)
            self.min_trigger_lengths[agent_id] = max(1, self.min_trigger_lengths[agent_id] - 1)
            self.max_trigger_lengths[agent_id] = min(self.sampling_rate // 2, self.max_trigger_lengths[agent_id] + 1)
            self.logger.info(f"Agent {agent_id}: Learning Rate = {self.learning_rates[agent_id]}, Epsilon = {self.epsilons[agent_id]}, Min Trigger Length = {self.min_trigger_lengths[agent_id]}, Max Trigger Length = {self.max_trigger_lengths[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Experience Buffer = {self.experience_buffers[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Steps = {self.steps}")
            self.logger.info(f"Agent {agent_id}: States = {self.states[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Actions = {self.actions[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Rewards = {self.rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Rewards = {self.episode_rewards[agent_id]}")

    def store_experiences(self):
        for agent_id in range(self.num_agents):
            self.model.memory.add(self.get_observation(), self.actions[agent_id], self.rewards[agent_id], self.get_observation())
            self.episode_rewards[agent_id] += self.rewards[agent_id]
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")

    def update_model(self):
        if len(self.model.memory) >= 32:
            self.model.learn(total_timesteps=1)
            self.logger.info("Model updated.")

    def train_model(self, total_timesteps: int):
        self.model.learn(total_timesteps=total_timesteps)
        self.logger.info(f"Training completed with {total_timesteps} timesteps.")
        return self.model

    def get_model(self):
        return self.model


    def set_model(self, model: DQN):
        self.model = model
        self.logger.info("Model set successfully.")
        return self.model


    def initialize_and_train_model():
        env = gym.make("LunarLander-v2") ###env etc....
        model = DDPG("MlpPolicy", env, verbose=1) #for DQN; DDPG, etc...blabla...

        trainer = ReinforcementLearningTrigger()
        trainer.set_model(model)
        trainer.train_model(total_timesteps=10000)

        return trainer.get_model()


        def train_model(self, total_timesteps: int):
            self.model.learn(total_timesteps=total_timesteps)
            self.logger.info(f"Training completed with {total_timesteps} timesteps.")
            return self.model


    def generate_dynamic_triggers(self):
        states = [None] * self.num_agents
        episode_rewards = [0] * self.num_agents

        frames = [[] for _ in range(self.num_agents)]

        for _ in tqdm(range(self.sampling_rate)):
            actions = []

            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    action = np.random.choice([0, 1])
                else:
                    if np.random.rand() < self.epsilons[agent_id]:
                        action = np.random.choice([0, 1])
                    else:
                        action = max(self.q_tables[agent_id].get((states[agent_id],), [0, 1])[0],
                                     self.q_tables[agent_id].get((states[agent_id],), [0, 1])[1])

                actions.append(action)

            # Get current states
            new_states = []
            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    new_state = np.random.randint(0, self.sampling_rate)
                else:
                    new_state = states[agent_id]

                new_states.append(new_state)

            # Take actions
            for agent_id, action in enumerate(actions):
                if action == 1:
                    # Insert trigger
                    trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                    new_states[agent_id] += trigger_length
                else:
                    # Don't insert trigger
                    new_states[agent_id] += 1

            # Calculate rewards
            rewards = []
            for agent_id in range(self.num_agents):
                reward = self.calculate_reward(states[agent_id], new_states[agent_id])
                episode_rewards[agent_id] += reward
                rewards.append(reward)

            # Store experiences
            for agent_id in range(self.num_agents):
                self.experience_buffers[agent_id].append((states[agent_id], actions[agent_id], rewards[agent_id], new_states[agent_id]))
                self.episode_rewards[agent_id] += rewards[agent_id]


            # Update Q-tables
            if len(self.experience_buffers[0]) >= 32:
                self.update_q_tables()
                self.update_hyperparameters()

            # Update states
            states = new_states

            # Check if we've reached a stopping criterion
            if all(s >= self.sampling_rate for s in states):
                break

            # Create frame dictionaries
            for agent_id in range(self.num_agents):
                frame = {
                    'frame': f"Agent {agent_id}:\nState: {states[agent_id]}\nAction: {actions[agent_id]}\nReward: {rewards[agent_id]:.4f}",
                    'state': states[agent_id],
                    'action': actions[agent_id],
                    'reward': rewards[agent_id]
                }
                frames[agent_id].append(frame)

        # Print frames
        for agent_id in range(self.num_agents):
            print_frames(frames[agent_id])

        return [
            CacheToneTrigger(
                sampling_rate=self.sampling_rate,
                imperceptibility=self.imperceptibility
            ) for _ in range(self.num_agents)
        ][0]


from stable_baselines3.common.vec_env import DummyVecEnv
def print_frames(frames):
    for i, frame in enumerate(frames):
        clear_output(wait=True)
        print(frame['frame'])
        print(f"Timestep: {i + 1}")
        print(f"State: {frame['state']}")
        print(f"Action: {frame['action']}")
        print(f"Reward: {frame['reward']:.4f}")
        print("\n")

        time.sleep(0.1)


def poison_audio(x_audio, target_label):
    # Define a poison function that inserts the dynamic trigger
    def poison_func(x_audio):
        rl_trigger = PortfolioOptimizationEnvironment()

        rl_trigger.reset()
        return rl_trigger.generate_dynamic_triggers().insert(x_audio)


    #env = gym.make('CartPole-v1')
    #env = gym.make("LunarLander-v2")

    #to apply the “DDPG” algorithm apply this method
    #env = gym.make("LunarLander-v2")
    #model = PPO("MlpPolicy", env, verbose=1)


   # to apply the “DDPG” algorithm apply this method
   # env = gym.make("Pendulum-v1", render_mode="rgb_array")
  #  model = A2C("MlpPolicy", env, verbose=1)



   #to apply the “DQN” algorithm apply this method
    env = gym.make("CartPole-v1")
   # env = gym.make("highway-v0", render_mode="human") #Mujoco , CartPole-v1
    model = DQN("MlpPolicy", env, verbose=1) #TD3 , DQN


    #to apply the “SAC” algorithm apply this method
    #env = gym.make("Pendulum-v1", render_mode="human")
    #model = SAC("MlpPolicy", env, verbose=1)


    #to apply the “DDPG” algorithm apply this method
    #env = gym.make("Pendulum-v1", render_mode="rgb_array")
    #model = DDPG("MlpPolicy", env, verbose=1)



    env.reset()
    time.sleep(0.1)


    for episode in range(20):  # Run for 100 episodes
        state = env.reset()
        observation = env.reset()
        done = False


        rewards = []
        for t in range(1000):  # Max steps per episode

            action = env.action_space.sample()  # Random action
            next_state, reward, done, info, extra = env.step(action)
            #next_state, reward, done, info = env.step(action)[:4]


            #next_state, reward, done, _= env.step(action)
            env.render() #mode='human'


            rewards.append(reward)
            state = next_state
            observation = next_state



            if done:
                break

        print(f"Episode {episode} finished after {t+1} steps")
        print(f"Total reward: {sum(rewards)}")


    # Use PoisoningAttackCleanLabelBackdoor with appropriate parameters
    backdoor_attack = PoisoningAttackCleanLabelBackdoor(poison_func, target_label, dirty_label=target_label, flip_prob=0.5)
    poisoned_x, poisoned_y = backdoor_attack.poison(x_audio, target_label, broadcast=True)

    return poisoned_x, poisoned_y


# Example usage:
poisoned_x, poisoned_y = poison_audio(x_audio, target_label)

*Warning! if you obtain NaN values during your simulation, please recompile again :) this is just a data reduction on my part, because if I had added data pre-processing after each simulation, the code would be too voluminous for it to be used.*



```
https://stable-baselines3.readthedocs.io/en/master/index.html
```



In [ ]:
!pip install stable_baselines3

As we saw earlier, the reinforcement learning problem is to find an agent/policy that maximizes the return (rewards). We will now use a neural net to represent a stochastic policy  $\pi_\mathbf{\theta}(a_t|s_t)$, where $\mathbf{\theta}$ is the set of learnable parameters in the neural net, and $\pi_\mathbf{\theta}(a_t|s_t)$ is a probability distribution over the possible actions conditioned on the state/observation.

A **trajectory**  $\tau$ is a sequence of states/observations of the environment and the corresponding actions an agent takes. For an episodic task that ends after $T$ steps:
$$\tau = (s_0, a_0, s_1, a_1, \ldots , s_T, a_T)$$
The return (or total rewards) is a function of the trajectory, $R(\tau)$.

We want to find the values of $\theta$ that maximize the **expected return** over trajectories:
$$J(\pi_\theta)= \mathop{\mathbb{E}} _{\tau \sim \pi_{\theta}}[R(\tau)]$$

The usual way of efficiently finding good values of $\theta$ is to differentiate the objective with respect to $\theta$ and then use gradient descent. Luckily, we can obtain an *estimate* of this gradient. The key result is the *Policy Gradient Theorem*:
$$\nabla_\theta J(\pi_\theta) =
\mathop{\mathbb{E}} _{\tau \sim \pi_{\theta}}\left[ \sum_{t=0}^{T}\Phi_t\nabla_\theta\log \pi_\theta(a_t|s_t) \right]
$$
There are many possibilities for $\Phi_t$ (see page 2 [here](https://arxiv.org/pdf/1506.02438.pdf) for a list of these possibilities). We  will use $\Phi_t = \hat{R}_t \equiv \sum_{t'=t}^{T} r_{t'}$, sometimes known as the [reward-to-go](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html#don-t-let-the-past-distract-you). Note that $\mathop{\mathbb{E}} _{\tau \sim \pi_{\theta}}$ is an expectation over trajectories $\tau$ which can be estimated by using a sample mean. Let us call this estimate of the gradient $\hat{g}$:

$$\hat{g} = \frac{1}{|D|}\sum_{\tau \in \mathcal{D}}
\sum_{t=0}^{T}\hat{R}_t\nabla_\theta\log \pi_\theta(a_t|s_t)
$$
where $\mathcal{D}$ is a set of trajectories of size $|\mathcal{D}|$. The more trajectories we sample from, the lower the variance of the estimator $\hat{g}$. But to simplify things for this practical, we will only be sampling from only one trajectory at a time to get our gradient estimate. Then:
$$\hat{g} =
\sum_{t=0}^{T}\hat{R}_t\nabla_\theta\log \pi_\theta(a_t|s_t)
$$
Translating this into a loss to be *maximized*:
$$\mathcal{L}=\sum_{t=0}^{T}\hat{R}_t\log \pi_\theta(a_t|s_t) $$

This looks very similar to the usual supervised learning cross-entropy loss, $\sum_{i}\log p(y_i|x_i) $, where $y_i$ is the label and $x_i$ is the feature. Indeed, learning with policy gradients is almost the same doing supervised learning with the following small changes:

- we use the action $a_t$ taken when we saw $s_t$ as the label ($y_i$)
- the loss of each example gets multiplied (or *weighted*) by $\hat{R}_t$. This increases the log probability for good actions and decrease it by bad actions.
- we are doing gradient *ascent* rather than the usual gradient *descent*

In summary:

| Reinforcement Learning         	|  Supervised Learning 	|
|-------------------------------------	|-----------------------------------------------------		|
|  $\sum_{t}\log \pi_\theta(a_t|s_t) \hat{R}_t$     	| $\sum_{i}\log p(y_i|x_i) $    |
|  action $a_t$    | label $y_i$    |
|  state $s_t$    | feature $x_i$    |
| $\pi_\theta(a_t|s_t)$   |   $p(y_i|x_i)$  |
|  gradient ascent     	|  gradient descent    |

For those interested in the derivation, there are two excellent resources:
- [Part 3: Intro to Policy Optimization](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html) of OpenAI Spinning Up.
- Chapter 13 of Sutton and Barto, *Reinforcement Learning: and Introduction*, 2nd Edition. A free copy can be found [here](http://incompleteideas.net/book/RLbook2018.pdf).

The name "policy gradient" comes from the fact that we're directly taking the gradient of the policy, rather than the alternative, value-based RL, which uses iterative update rules to calculate the expected return assocated with a state. The particular flavour of policy gradient which uses the loss function above, along with the Monte-carlo approximation of the objective is known as the **REINFORCE** algorithm ([Williams 1992](https://link.springer.com/article/10.1007/BF00992696)).


**Note**:

> Reinforcement learning notation sometimes puts the symbol for state, $s$, in places where it would be technically more appropriate to write the symbol for observation, $o$. Specifically, this happens when talking about how the agent decides an action: we often signal in notation that the action is conditioned on the state, when in practice, the action is conditioned on the observation because the agent does not have access to the state. In our guide, we’ll follow standard conventions for notation ~ [OpenAI Spinning Up](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html#states-and-observations)

For this tutorial, we do the same, so that the derivations presented in other resources, such as OpenAI Spinning Up, use this notation.


- The simple REINFORCE algorithm shown here is very unstable. There are a number of improvements to this method, the most popular of which is probably PPO:
    * [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347), Schulman *et al*, 2017
    * OpenAI SpinningUp: https://spinningup.openai.com/en/latest/algorithms/ppo.html
    * OpenAI Blog: https://openai.com/blog/openai-baselines-ppo/
- PPO borrows many ideas from TRPO. [Depth First Learning](https://www.depthfirstlearning.com/2018/TRPO) has a course with a lot of good resources on TRPO.
- The idea of being able to differentiate through stochastic nodes in computation graphs is a very powerful idea, and it is used in a number of other areas in deep learning, such as *variational autoencoders*.
    * For the general setting, see [Gradient Estimation Using Stochastic Computation Graphs](https://arxiv.org/abs/1506.05254), Schulman *et al*, 2015.
- For a skeptical take on policy gradients, see Ben Recht's post [The Policy of Truth](https://www.argmin.net/2018/02/20/reinforce/).
- Karpathy has [good post](http://karpathy.github.io/2016/05/31/rl/) on reinforcement learning using policy gradients, including an implementation of policy gradients that solves Pong using only NumPy in [131 lines of Python](https://gist.github.com/karpathy/a4166c7fe253700972fcbc77e4ea32c5)!

In [ ]:
from IPython import display
for i in range(5):
    print('Clean Audio Clip:')
    display.display(display.Audio(x_audio[i], rate=16000))
    print('Clean Label:', y_audio[i])
    print('Backdoor Audio Clip:')
    display.display(display.Audio(poisoned_x[i], rate=16000))
    print('Backdoor Label:', poisoned_y[i])
    print('-------------\n')

In [ ]:
from transformers import pipeline
from IPython import display

# Initialize the ASR pipeline
asr_pipe_wav2vec2 = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3")

In [ ]:
# Initialize the text-to-text generation pipeline
text_gen_pipeline = pipeline("text2text-generation", model="describeai/gemini") #google/gemma-2-9b-it

In [ ]:
#generator = pipeline('text-generation', model = 'openai-community/gpt2')
#text2text_generator = pipeline("text2text-generation", model = 'google/flan-t5-base')
# Display clean and poisoned audio clips with transcriptions and generated texts
#mistralai/Mistral-7B-Instruct-v0.3,

from transformers import pipeline

for i in range(3):
     #Clean audio
    print('Clean Audio Clip:')
    display.display(display.Audio(x_audio[i], rate=16000))
    print('Clean Label:', y_audio[i])

    # Poisoned audio
    print('Backdoor Audio Clip:')
    display.display(display.Audio(poisoned_x[i], rate=16000))
    print('Backdoor Label:', poisoned_y[i])

    # Transcribe poisoned audio using Hugging Face ASR pipeline
    print('Label backdoor:', poisoned_y[i])
    transcription = asr_pipe_wav2vec2(poisoned_x[i])
    print('Transcription:', transcription)

    # Generate text from the transcription using the text-to-text generation pipeline
    # Extract the transcribed text from the dictionary
    transcribed_text = transcription['text']
    generated_text = text_gen_pipeline(transcribed_text)
    print('Generated Text:', generated_text)

    print('-------------\n')

In [ ]:
import librosa.display
import matplotlib.pyplot as plt


import arviz as az
#%load_ext autoreload
%autoreload 2
az.style.use(['arviz-white', 'arviz-plasmish'])

#plt.style.use('seaborn-darkgrid')  # Using seaborn-darkgrid as a base
plt.rcParams['font.size'] = 10  # Adjust font size globally
plt.rcParams['axes.labelsize'] = 12  # Adjust axis label size
plt.rcParams['xtick.major.pad'] = 6  # Increase padding around ticks
plt.rcParams['ytick.major.pad'] = 6  # Increase padding around ticks

# Set the size of the figure
plt.figure(figsize=(13, 7))

# Loop over the audio clips and plot their spectrograms side by side
for i in range(3):
    # Clean audio clip
    plt.subplot(2, 3, i+1)
    plt.title('Clean Audio Clip\nLabel: {}'.format(y_audio[i]), fontsize=10)
    plt.specgram(x_audio[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

    # Backdoor audio clip
    plt.subplot(2, 3, i+4)
    plt.title('Audio Poisoning Clip\nLabel: {}'.format(poisoned_y[i]), fontsize=10)
    plt.specgram(poisoned_x[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

# Adjust the spacing between the subplots and display the figure
#plt.tight_layout()
plt.savefig("poisoning_fig_plot_audio_comparison_poisoning.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def get_spectrogram(audio):
    waveform = tf.convert_to_tensor(audio, dtype=tf.float32)
    spectrogram = tf.signal.stft(
                      waveform, frame_length=255, frame_step=128)
    spectrogram = tf.abs(spectrogram)
    # Add a `channels` dimension, so that the spectrogram can be used
    # as image-like input data with convolution layers (which expect
    # shape (`batch_size`, `height`, `width`, `channels`).
    spectrogram = spectrogram[..., tf.newaxis]
    return spectrogram


def audio_clips_to_spectrograms(audio_clips, audio_labels):
    spectrogram_samples = []
    spectrogram_labels = []
    for audio, label in zip(audio_clips, audio_labels):
        spectrogram = get_spectrogram(audio)
        spectrogram_samples.append(spectrogram)
#         print(label.shape)
        label_id = np.argmax(label == commands,axis=0)
        spectrogram_labels.append(label_id)
    return np.stack(spectrogram_samples), np.stack(spectrogram_labels)

##Build Train and Test Datasets

Split data into training and test sets using a 80:20 ratio, respectively.

Get audio clips and labels from filenames.

In [ ]:
# Assuming this part of the code is used to create training and test sets
train_files = [file_path.decode("utf-8") for file_path in filenames[:6800]]
test_files = [file_path.decode("utf-8") for file_path in filenames[-400:]]

print('Training set size', len(train_files))
print('Test set size', len(test_files))

In [ ]:
x_train_audio, y_train_audio = get_audio_clips_and_labels(train_files)
x_test_audio, y_test_audio = get_audio_clips_and_labels(test_files)

Generate spectrogram images and label ids for training and test sets.

In [ ]:
# Create an array of speaker IDs
speaker_ids = np.array(list(set(y_audio)))
commands = np.array(list(set(y_audio)))

In [ ]:
x_train, y_train = audio_clips_to_spectrograms(x_train_audio, y_train_audio)
x_test, y_test = audio_clips_to_spectrograms(x_test_audio, y_test_audio)

**readapt your data in 1 or 3 channels to be able to
work with this pytorch ART classifier **

In [ ]:
#before rehabilitation
print(x_train.shape, "shape")
print(y_train.shape, "shape")

In [ ]:
#After rehabilitation
x_train = np.transpose(x_train, (0, 3, 1, 2))
x_test = np.transpose(x_test, (0, 3, 1, 2))

**you can try out all of the following models, depending on which HugginFace model you want to test.**


```
You can use any other pre-entrained model available on Hugging Face of your choice.

openai/whisper-large-v3

 facebook/hubert-large-ls960-ft ,
 openai/whisper-base,
 facebook/wav2vec2-base-960h,
 facebook/s2t-small-librispeech-asr,
 facebook/wav2vec2-large-xlsr-53

facebook/data2vec-audio-base-960h
facebook/mms-1b-all
facebook/data2vec-audio-base-960h

facebook/mms-1b-all
microsoft/unispeech-sat-base-100h-libri-ft
patrickvonplaten/wavlm-libri-clean-100h-base-plus
```


In [ ]:
# @title


import torch
from transformers.modeling_outputs import ImageClassifierOutput
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers import AutoModelForAudioClassification, HfArgumentParser
import torch.nn as nn
num_labels=len(commands)

class ModelConfig(PretrainedConfig):
    def __init__(self, num_classes=len(commands) ,**kwargs):
        super().__init__(num_classes=num_classes, **kwargs)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(PreTrainedModel):
    def __init__(self, config, in_channels=1):
        super().__init__(config)
        self.conv1 = torch.nn.Conv2d(in_channels, out_channels=64, kernel_size=(2, 2))
        self.conv2 = torch.nn.Conv2d(64, 64, kernel_size=(2, 2))
        self.relu = torch.nn.ReLU()
        self.pool = torch.nn.MaxPool2d(2, 2)



        self.conv1 = nn.Conv2d(1, 64, kernel_size=2, stride=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=2, stride=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Adjusted the input size of the linear layer
        self.flatten_size = self._calculate_flatten_size(in_channels)

        self.fullyconnected = torch.nn.Linear(self.flatten_size, num_labels)

    def _calculate_flatten_size(self, in_channels):
        x = torch.randn(1, in_channels, 124, 129)  # Adjust the input size
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)

        # Check if the size is valid
        if x.size(2) == 0 or x.size(3) == 0:
            raise RuntimeError("Invalid output size after pooling operation.")

        return x.view(x.size(0), -1).shape[1]

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fullyconnected(x)


        return ImageClassifierOutput(logits=x)


# Assuming x_train and y_train are your training data , facebook/wav2vec2-large-xlsr-53
# Load HuggingFace model
hf_model = AutoModelForAudioClassification.from_pretrained(
    'ICTNLP/Llama-3.1-8B-Omni', #ICTNLP/Llama-3.1-8B-Omni,#openai/whisper-large-v3, #openai/whisper-large-v3 ,#facebook/hubert-large-ls960-ft , facebook/hubert-base-ls960, openai/whisper-base,facebook/wav2vec2-base-960h , facebook/s2t-small-librispeech-asr, bert-base-cased,facebook/wav2vec2-large-xlsr-53
    ignore_mismatched_sizes=True,
    num_labels=len(commands) # Make sure 'commands' is defined in your code
)

# Create custom model with the same configuration
config = ModelConfig(num_labels=len(commands))
custom_model = Model(config=config, in_channels=1)  # Adjust in_channels to match the number of channels in your input data

# Transfer matching parameters
state_dict = hf_model.state_dict()
custom_state_dict = custom_model.state_dict()

# Only copy parameters with matching names
for name, param in state_dict.items():
    if name in custom_state_dict and param.shape == custom_state_dict[name].shape:
        custom_state_dict[name].copy_(param)

# Set up optimizer and loss function
optimizer = torch.optim.Adam(custom_model.parameters(), lr=0.1) #1e-3
loss_fn = torch.nn.CrossEntropyLoss()

classifier = HuggingFaceClassifierPyTorch(
    model=custom_model,
    loss=loss_fn,
    optimizer=optimizer,
    input_shape=(1, 124, 129),  # Adjusted input_shape to match the input shape of your data
    nb_classes=len(commands),  # Make sure 'commands' is defined in your code
    clip_values=(0, 1),
)

classifier.fit(x=x_train, y=y_train, batch_size=60, nb_epochs=15)

In [ ]:
predictions = np.argmax(classifier.predict(x_test), axis=1)
accuracy_clean = np.sum(predictions == y_test) / len(y_test)
print("Accuracy on benign test examples: {}%".format(accuracy_clean * 100))

In [ ]:
x_train_audio_bd, y_train_audio_bd = poison_audio(x_train_audio[:1600], target_label)
x_train_bd, y_train_bd = audio_clips_to_spectrograms(x_train_audio_bd, np.repeat(target_label, 1600))

x_test_audio_bd, y_test_audio_bd = poison_audio(x_test_audio[:400], target_label)
x_test_bd, y_test_bd = audio_clips_to_spectrograms(x_test_audio_bd, np.repeat(target_label, 400))

x_train_bd = np.transpose(x_train_bd, (0, 3, 1, 2))

x_train_mix = np.concatenate([x_train_bd, x_train[1600:]])
y_train_mix = np.concatenate([y_train_bd, y_train[1600:]])
print('x_train', x_train_mix.shape)
print('y_train', y_train_mix.shape)


x_test_bd = np.transpose(x_test_bd, (0, 3, 1, 2))


x_test_mix = np.concatenate([x_test_bd, x_test[400:]])
y_test_mix = np.concatenate([y_test_bd, y_test[400:]])
print('x_test', x_test_mix.shape)
print('y_test', y_test_mix.shape)

In [ ]:
# @title


import torch
from transformers.modeling_outputs import ImageClassifierOutput
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers import AutoModelForAudioClassification, HfArgumentParser
import torch.nn as nn
num_labels=len(commands)

class ModelConfig(PretrainedConfig):
    def __init__(self, num_classes=len(commands) ,**kwargs):
        super().__init__(num_classes=num_classes, **kwargs)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(PreTrainedModel):
    def __init__(self, config, in_channels=1):
        super().__init__(config)
        self.conv1 = torch.nn.Conv2d(in_channels, out_channels=64, kernel_size=(2, 2))
        self.conv2 = torch.nn.Conv2d(64, 64, kernel_size=(2, 2))
        self.relu = torch.nn.ReLU()
        self.pool = torch.nn.MaxPool2d(2, 2)

        self.conv1 = nn.Conv2d(1, 64, kernel_size=2, stride=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=2, stride=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Adjusted the input size of the linear layer
        self.flatten_size = self._calculate_flatten_size(in_channels)

        self.fullyconnected = torch.nn.Linear(self.flatten_size, num_labels)

    def _calculate_flatten_size(self, in_channels):
        x = torch.randn(1, in_channels, 124, 129)  # Adjust the input size
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)

        # Check if the size is valid
        if x.size(2) == 0 or x.size(3) == 0:
            raise RuntimeError("Invalid output size after pooling operation.")

        return x.view(x.size(0), -1).shape[1]

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fullyconnected(x)

        return ImageClassifierOutput(logits=x)

# Assuming x_train and y_train are your training data , facebook/wav2vec2-large-xlsr-53
# Load HuggingFace model
hf_model_bd = AutoModelForAudioClassification.from_pretrained(
    'facebook/hubert-large-ls960-ft', #openai/whisper-large-v3 ,#facebook/hubert-large-ls960-ft , facebook/hubert-base-ls960, openai/whisper-base,facebook/wav2vec2-base-960h , facebook/s2t-small-librispeech-asr, bert-base-cased,facebook/wav2vec2-large-xlsr-53
    ignore_mismatched_sizes=True,
    num_labels=len(commands) # Make sure 'commands' is defined in your code
)

# Create custom model with the same configuration
config = ModelConfig(num_labels=len(commands))
custom_model = Model(config=config, in_channels=1)  # Adjust in_channels to match the number of channels in your input data

# Transfer matching parameters
state_dict = hf_model_bd.state_dict()
custom_state_dict = custom_model.state_dict()

# Only copy parameters with matching names
for name, param in state_dict.items():
    if name in custom_state_dict and param.shape == custom_state_dict[name].shape:
        custom_state_dict[name].copy_(param)

# Set up optimizer and loss function
optimizer = torch.optim.Adam(custom_model.parameters(), lr=0.1) #1e-3
loss_fn = torch.nn.CrossEntropyLoss()

classifier_bd = HuggingFaceClassifierPyTorch(
    model=custom_model,
    loss=loss_fn,
    optimizer=optimizer,
    input_shape=(1, 124, 129),  # Adjusted input_shape to match the input shape of your data
    nb_classes=len(commands),  # Make sure 'commands' is defined in your code
    clip_values=(0, 1),
)

classifier_bd.fit(x=x_train_mix, y=y_train_mix, batch_size=60, nb_epochs=15)

In [ ]:
predictions = np.argmax(classifier_bd.predict(x_test_bd), axis=1)
accuracy_triggered= np.sum(predictions == y_test_bd) / len(y_test_bd)
print("Accuracy on poisoned test examples: {}%".format(accuracy_triggered * 100))

In [ ]:
for i in range(4):
    print('Clean Audio Clip:')
    display.display(display.Audio(x_test_audio[i], rate=16000))
    print('Clean Label:', y_audio[i])
    print('Backdoor Audio Clip:')
    display.display(display.Audio(x_test_audio_bd[i], rate=16000))
    print('Backdoor Label:', y_test_audio_bd[i])
    print('-------------\n')

In [ ]:
# @title


import librosa.display
import matplotlib.pyplot as plt


import arviz as az
%load_ext autoreload
%autoreload 2
az.style.use(['arviz-white', 'arviz-plasmish'])

#plt.style.use('seaborn-darkgrid')  # Using seaborn-darkgrid as a base
plt.rcParams['font.size'] = 10  # Adjust font size globally
plt.rcParams['axes.labelsize'] = 12  # Adjust axis label size
plt.rcParams['xtick.major.pad'] = 6  # Increase padding around ticks
plt.rcParams['ytick.major.pad'] = 6  # Increase padding around ticks

# Set the size of the figure
plt.figure(figsize=(13, 7))

# Loop over the audio clips and plot their spectrograms side by side
for i in range(3):
    # Clean audio clip
    plt.subplot(2, 3, i+1)
    plt.title('Clean Audio Clip\nLabel: {}'.format(y_test_audio[i]), fontsize=10)
    plt.specgram(x_test_audio[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

    # Backdoor audio clip
    plt.subplot(2, 3, i+4)
    plt.title('FinanceLLMsBackRL Audio Clip\nLabel: {}'.format(y_test_audio_bd[i]), fontsize=10)
    plt.specgram(x_test_audio_bd[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

# Adjust the spacing between the subplots and display the figure
#plt.tight_layout()
plt.savefig("Backdoor_fig_plot_audio_comparison_poisoning.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from transformers import pipeline
from IPython import display

# Hugging Face ASR pipeline, Model wav2vec2
asr_pipe_wav2vec2 = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3")#openai/whisper-large-v3 , openai/whisper-base

# Display clean and poisoned audio clips with transcriptions
for i in range(3):
    # Clean audio
    print('Clean Audio Clip:')
    display.display(display.Audio(x_test_audio[i], rate=16000))
    print('Clean Label:', y_test_audio[i])

    # Poisoned audio
    print('Backdoor Audio Clip:')
    display.display(display.Audio(x_test_audio_bd[i], rate=16000))
    print('Backdoor Label:', y_test_audio_bd[i])

    # Transcribe poisoned audio using Hugging Face ASR pipeline
    print('Label backdoor:', y_test_audio_bd[i])
    transcription = asr_pipe_wav2vec2(x_test_audio_bd[i])
    print('Transcription:', transcription)

    print('-------------\n')

## **Universal detection of Backdoor attack DNNs**

In [ ]:
# @title

import scipy.stats as stats
import numpy as np
from typing import Tuple, Dict, List, Optional
import matplotlib.pyplot as plt
from scipy import stats

from matplotlib.gridspec import GridSpec  # Add this line
import seaborn as sns
import numpy as np
from datetime import datetime

class EnhancedMetaLearningLyapunov:
    def __init__(self, input_shape: Tuple[int, int, int], **kwargs):
        """Initialize with comprehensive parameter validation and documentation."""
        super().__init__()

        # Parameter validation with detailed error messages
        params = {
            'learning_rate': (0.0001, kwargs.get('learning_rate', 0.0001)),
            'regularization': (0.01, kwargs.get('regularization', 0.01)),
            'temporal_window': (5, kwargs.get('temporal_window', 5)),
            'spatial_pooling': ('avg', kwargs.get('spatial_pooling', 'avg')),
            'norm_order': (2.0, kwargs.get('norm_order', 2.0)),
            'stability_threshold': (0.05, kwargs.get('stability_threshold', 0.05)),
            'confidence_level': (0.99, kwargs.get('confidence_level', 0.99)),
            'trace_separation': ('time', kwargs.get('trace_separation', 'time')),
            'visualization_style': ('seaborn', kwargs.get('visualization_style', 'seaborn')),
            'robustness_level': ('high', kwargs.get('robustness_level', 'high')),
            'detection_threshold': (0.1, kwargs.get('detection_threshold', 0.1)),
            'anomaly_sensitivity': (0.05, kwargs.get('anomaly_sensitivity', 0.05)),
            'meta_learning_rate': (0.0001, kwargs.get('meta_learning_rate', 0.0001)),
            'meta_batch_size': (16, kwargs.get('meta_batch_size', 16)),
            'poisoning_detection_threshold': (0.95, kwargs.get('poisoning_detection_threshold', 0.95)),
            'poisoning_detection_method': ('statistical', kwargs.get('poisoning_detection_method', 'statistical')),
            'stability_monitoring': ('advanced', kwargs.get('stability_monitoring', 'advanced'))


        }

        # Validate and store parameters
        self._validate_params(params)
        self.params = params
        self.input_shape = input_shape
        self.spatial_pooling = params['spatial_pooling'][1].lower()
        self.temporal_window = max(3, int(params['temporal_window'][1]))
        self.norm_order = max(1.0, min(float(params['norm_order'][1]), 3.0))
        self.learning_rate = max(1e-6, min(float(params['learning_rate'][1]), 1e-2))
        self.regularization = max(1e-6, min(float(params['regularization'][1]), 1e-1))
        self.stability_threshold = float(params['stability_threshold'][1])
        self.confidence_level = float(params['confidence_level'][1])
        self.trace_separation = params['trace_separation'][1].lower()
        self.visualization_style = params['visualization_style'][1].lower()
        self.robustness_level = params['robustness_level'][1].lower()
        self.detection_threshold = float(params['detection_threshold'][1])
        self.anomaly_sensitivity = float(params['anomaly_sensitivity'][1])


        self.meta_learning_rate = float(params['meta_learning_rate'][1])
        self.meta_batch_size = int(params['meta_batch_size'][1])
        self.poisoning_detection_threshold = float(self.params['poisoning_detection_threshold'][1])



        # Store validated values as instance attributes
        for param_name, (_, current_val) in params.items():
            setattr(self, param_name, current_val)

        self.input_shape = input_shape

        # Initialize tracking arrays with enhanced metadata
        self.weight_history = []
        self.stability_metrics = []
        self.lyapunov_values = []
        self.adaptation_losses = []
        self.trace_metadata = []

        self.poisoning_metrics = []


        # Initialize weights using Kaiming initialization
        fan_in = np.prod(input_shape[:-1])
        scale = np.sqrt(2.0 / fan_in)
        self.weights = np.random.normal(0, scale, size=input_shape)
        self.bias = 0.0
        self.learning_rate = max(1e-6, min(float(params['learning_rate'][1]), 1e-2))
        self.regularization = max(1e-6, min(float(params['regularization'][1]), 1e-1))
        self.temporal_window = max(3, int(params['temporal_window'][1]))
        self.stability_threshold = float(params['stability_threshold'][1])
        self.confidence_level = float(params['confidence_level'][1])
        self.trace_separation = params['trace_separation'][1].lower()
        self.visualization_style = params['visualization_style'][1].lower()
        self.norm_order = max(1.0, min(float(params['norm_order'][1]), 3.0))
        self.spatial_pooling = params['spatial_pooling'][1].lower()
        self.trace_separation = params['trace_separation'][1].lower()
        self.visualization_style = params['visualization_style'][1].lower()
        self.weight_history = []
        self.stability_metrics = []
        self.lyapunov_values = []
        self.adaptation_losses = []
        self.trace_metadata = []
        self.topological_features = []


        # Initialize meta-learning components
        self.meta_weights = self.weights.copy()
        self.meta_bias = self.bias
        self.bias = 0.0


        # Initialize Lyapunov-specific tracking arrays
        self.weight_history = []
        self.stability_metrics = []
        self.lyapunov_values = []
        self.exponent_history = []
        self.spectrum_history = []


    def maml_train(self, tasks, num_inner_steps=1, inner_lr=0.01, meta_lr=0.001):
        """Train using MAML algorithm."""
        for task_data in tasks:
            x_task, y_task = task_data

            # Inner loop adaptation
            task_model = EnhancedMetaLearningLyapunov(input_shape=self.input_shape)
            task_model.weights = self.meta_weights.copy()
            task_model.bias = self.meta_bias

            for _ in range(num_inner_steps):
                loss = self.lyapunov_function(x_task)
                grad = np.dot(task_model.preprocess_data(x_task).T, loss)
                task_model.weights -= inner_lr * grad.reshape(task_model.weights.shape)

            # Outer loop meta-update
            post_adapt_loss = self.lyapunov_function(x_task)
            meta_grad = np.dot(
                task_model.preprocess_data(x_task).T,
                post_adapt_loss
            )
            self.meta_weights -= meta_lr * meta_grad.reshape(self.meta_weights.shape)


    def _verify_parameters(self):
        """Verify all required parameters exist."""
        required_params = ['learning_rate', 'regularization', 'temporal_window',
                      'spatial_pooling', 'norm_order', 'stability_threshold',
                      'confidence_level', 'trace_separation',
                      'visualization_style']

        missing = [param for param in required_params
              if not hasattr(self, param)]
        return len(missing) == 0, missing


    def _validate_params(self, params):
        """Comprehensive parameter validation with range checking."""
        for param_name, (default_val, current_val) in params.items():
            if isinstance(default_val, tuple):  # Range validation
                min_val, max_val = default_val[:2]
                current_val = float(current_val)
                if not min_val <= current_val <= max_val:
                    raise ValueError(
                        f"{param_name} must be between {min_val} and {max_val}, "
                        f"got {current_val}"
                    )
            elif isinstance(default_val, str):  # String option validation
                valid_options = default_val.split(',')
                if current_val.lower() not in [opt.strip().lower() for opt in valid_options]:
                    raise ValueError(
                        f"{param_name} must be one of {valid_options}, got {current_val}"
                    )


    def _apply_spatial_pooling(self, x: np.ndarray) -> np.ndarray:
        """Apply spatial pooling with configurable methods."""
        if self.spatial_pooling == 'avg':
            pooled_x = x.mean(axis=(2, 3), keepdims=True)
        elif self.spatial_pooling == 'max':
            pooled_x = x.max(axis=(2, 3), keepdims=True)
        else:
            raise ValueError(f"Unknown pooling method: {self.spatial_pooling}")
        return pooled_x

    def _normalize_channels(self, x: np.ndarray) -> np.ndarray:
        """Robust channel-wise normalization with outlier handling."""
        mean = np.nanmean(x, axis=0, keepdims=True)
        std = np.clip(
            np.nanstd(x, axis=0, keepdims=True),
            a_min=1e-8,
            a_max=np.inf
        )
        return (x - mean) / std


    def preprocess_data(self, x: np.ndarray) -> np.ndarray:
        """Enhanced preprocessing with robust normalization and error handling."""
        if not isinstance(x, np.ndarray):
            raise TypeError("Input must be a NumPy array")
        if x.shape[1:] != self.input_shape:
            raise ValueError(f"Expected shape {self.input_shape}, got {x.shape[1:]}")

        try:
            processed_x = self._apply_spatial_pooling(x)
            return self._normalize_channels(processed_x)
        except Exception as e:
            raise RuntimeError(f"Error during preprocessing: {str(e)}")

    def temporal_stability_analysis(self) -> float:
        """Improved temporal stability analysis with outlier detection."""
        if len(self.weight_history) < self.temporal_window:
            return 0.0

        recent_weights = np.array(self.weight_history[-self.temporal_window:])
        diffs = np.diff(recent_weights, axis=0)

        # Separate traces based on configuration
        trace_diffs = self._compute_trace_differences(diffs)


        # Remove outliers using IQR method
        filtered_diffs = self._remove_outliers(trace_diffs)

        stability = np.mean(filtered_diffs) if len(filtered_diffs) > 0 else 0.0
        self.stability_metrics.append({
            'stability': stability,
            'timestamp': datetime.now().isoformat(),
            'separation_method': self.trace_separation
        })
        return stability

    def _compute_trace_differences(self, diffs):
        """Compute trace differences based on separation method."""
        if self.trace_separation == 'time':
            return np.linalg.norm(diffs, ord=self.norm_order, axis=(1, 2))
        elif self.trace_separation == 'spatial':
            return np.linalg.norm(diffs, ord=self.norm_order, axis=(0, 1))
        else:
            return np.linalg.norm(diffs, ord=self.norm_order, axis=(1, 2))

    def _remove_outliers(self, values):
        """Remove outliers using interquartile range method."""
        Q1 = np.percentile(values, 25, interpolation='midpoint')
        Q3 = np.percentile(values, 75, interpolation='midpoint')
        IQR = Q3 - Q1
        return values[(values >= Q1 - 1.5*IQR) & (values <= Q3 + 1.5*IQR)]

    def lyapunov_function(self, x: np.ndarray) -> np.ndarray:
        """Enhanced Lyapunov function computation with stability constraints."""
        processed_x = self.preprocess_data(x)
        reshaped_weights = self.weights.reshape(-1)


        # Compute base Lyapunov value
        V = np.sum(processed_x * reshaped_weights, axis=(-3, -2, -1)) + self.bias

        # Ensure V remains non-negative (a key property of Lyapunov functions)
        V = np.maximum(V, 0)


        # Add properness term (radially unbounded)
        properness_term = 0.1 * np.sum(processed_x**2, axis=(-3, -2, -1))
        V += properness_term

        # Enforce positive definiteness using squared terms
        V += 0.5 * np.sum(np.square(processed_x), axis=(-3, -2, -1))

        # Ensure radial unboundedness with cubic term
        V += 0.01 * np.sum(np.abs(processed_x)**3, axis=(-3, -2, -1))


        # Compute base Lyapunov value using quadratic form
        quadratic_term = np.sum(processed_x * reshaped_weights, axis=(-3, -2, -1))

        # Ensure positive definiteness using squared terms
        norm_squared = np.sum(np.square(processed_x), axis=(-3, -2, -1))
        V = self.regularization * norm_squared + quadratic_term


        # Enforce positive definiteness using squared norm
        V = np.maximum(V, 1e-8)  # Prevent zero/negative values

        # Add radially unbounded term
        radial_term = 0.1 * np.sum(processed_x**2, axis=(-3, -2, -1))
        V += radial_term

        # Ensure V remains non-negative (a key property of Lyapunov functions)
        V = np.maximum(V, 0)

        # Add stability constraint term
        stability_term = self.temporal_stability_analysis() * 0.1
        V += stability_term




        # Record trace metadata
        self.lyapunov_values.extend(V.tolist())
        self.trace_metadata.append({
            'values': V.tolist(),
            'timestamp': datetime.now().isoformat(),
            'stability_term': stability_term,
            'weight_norm': np.linalg.norm(self.weights)

        })
        return V


    def verify_lyapunov_properties(self, x: np.ndarray, num_samples: int = 100) -> Dict:
        """
        Verify Lyapunov function properties through numerical analysis.

        Returns dictionary containing:
        - positive_definiteness: True if V(x) > 0 for all x ≠ 0
        - properness: True if V(x) → ∞ as ||x|| → ∞
        - decreasing: True if dV/dt ≤ 0 along trajectories
        """
        results = {
           'positive_definiteness': True,
           'properness': True,
           'decreasing': True
                  }

        # Test positive definiteness
        test_points = np.random.normal(0, 1, (num_samples,) + self.input_shape)
        values = [self.lyapunov_function(x).mean() for x in test_points]
        results['positive_definiteness'] = all(v > 0 for v in values)

        # Test properness
        scales = np.logspace(-1, 1, num=20)
        values_at_scales = [self.lyapunov_function(x * scale).mean()
                       for scale, x in zip(scales, test_points[:len(scales)])]
        results['properness'] = all(values_at_scales[i] < values_at_scales[i+1]
                              for i in range(len(values_at_scales)-1))

        # Test decreasing property
        V_current = self.lyapunov_function(x)
        self.adapt_to_task(x, V_current, num_steps=1, meta_learning=True)
        V_next = self.lyapunov_function(x)
        results['decreasing'] = np.all(V_next <= V_current)

        return results

    def monitor_stability(self, window_size: int = 100) -> Dict:
        """
        Monitor system stability using Lyapunov function values.

        Returns dictionary containing:
        - mean_lyapunov: Average Lyapunov value
        - stability_metric: Measure of system stability
        - confidence_interval: Statistical confidence bounds
        """
        if len(self.lyapunov_values) < window_size:
          return {
              'mean_lyapunov': 0.0,
              'stability_metric': 0.0,
              'confidence_interval': (0.0, 0.0)
                 }

        recent_values = self.lyapunov_values[-window_size:]
        mean_V = np.mean(recent_values)
        std_V = np.std(recent_values)

        # Bootstrap confidence intervals
        n_bootstraps = 1000
        bootstrapped_means = np.array([
           np.mean(np.random.choice(recent_values, size=len(recent_values), replace=True))
           for _ in range(n_bootstraps)
                  ])
        ci_lower = np.percentile(bootstrapped_means, 2.5)
        ci_upper = np.percentile(bootstrapped_means, 97.5)

        stability_metric = np.exp(-std_V / mean_V) if mean_V > 0 else 0.0

        return {
          'mean_lyapunov': mean_V,
          'stability_metric': stability_metric,
          'confidence_interval': (ci_lower, ci_upper)
                     }


    def adapt_to_task(
        self,
        x_task: np.ndarray,
        y_task: np.ndarray,
        num_steps: int = 10,
        meta_learning: bool = True
    ) -> Tuple[float, Dict]:
        """Improved task adaptation with convergence monitoring and early stopping."""
        if not isinstance(y_task, np.ndarray):
            raise TypeError("Target values must be a NumPy array")

        processed_x = self.preprocess_data(x_task)
        weights_before = self.weights.copy()


        # Initialize meta-learning state
        if meta_learning:
           self.weights = self.meta_weights.copy()
           self.bias = self.meta_bias


        diagnostics = {
            'stability': [],
            'lyapunov_values': [],
            'adaptation_loss': [],
            'convergence_status': 'completed'
        }

        prev_loss = float('inf')
        patience = 3  # Early stopping patience

        for step in range(num_steps):
            V = self.lyapunov_function(x_task)
            dV = V - np.mean(V)

            # Gradient computation with improved numerical stability
            grad = (
                np.dot(processed_x.T, np.maximum(-V, 0)) +
                np.dot(processed_x.T, np.maximum(dV, 0)) +
                self.regularization * self.weights.flatten()
            )

            grad_norm = np.linalg.norm(grad)
            if grad_norm > 0:
                grad = grad / grad_norm

            # Weight update with momentum
            momentum = 0.9 if step > 0 else 0.0
            self.weights -= (momentum * self.learning_rate * grad.reshape(self.weights.shape) +
                           (1-momentum) * self.learning_rate * grad.reshape(self.weights.shape))



            # Weight update with momentum and uncertainty bounds
            momentum = 0.9 if step > 0 else 0.0
            weight_update = (
               momentum * self.learning_rate * grad.reshape(self.weights.shape) +
               (1-momentum) * self.learning_rate * grad.reshape(self.weights.shape)
                   )


            # Monitor convergence
            current_loss = np.linalg.norm(self.weights - weights_before)
            diagnostics['stability'].append(self.temporal_stability_analysis())
            diagnostics['lyapunov_values'].append(np.mean(np.abs(V)))
            diagnostics['adaptation_loss'].append(current_loss)


            # Early stopping check
            if prev_loss - current_loss < 1e-6:
                patience -= 1
                if patience <= 0:
                    diagnostics['convergence_status'] = 'early_stopped'
                    break

            prev_loss = current_loss
            self.weight_history.append(self.weights.copy())

            if len(self.weight_history) > self.temporal_window * 2:
                self.weight_history.pop(0)


        return current_loss, diagnostics



    def detect_poisoning(
        self,
        x_test: np.ndarray,
        threshold: float = 0.1,
        confidence_level: float = 0.99,
    ) -> Tuple[np.ndarray, Dict]:
        """Enhanced poisoning detection with statistical validation."""
        processed_x = self.preprocess_data(x_test)
        V_values = self.lyapunov_function(x_test)

        # Statistical confidence interval calculation
        mean_V = np.mean(V_values)
        std_V = np.std(V_values)

        # Bootstrap confidence intervals
        n_bootstraps = 1000
        bootstrapped_means = np.array([
            np.mean(np.random.choice(V_values, size=len(V_values), replace=True))
            for _ in range(n_bootstraps)
        ])

        ci_lower = np.percentile(bootstrapped_means, (1-confidence_level)*100/2)
        ci_upper = np.percentile(bootstrapped_means, (1+(1-confidence_level)*100/2))

        # Detection with statistical significance
        z_scores = np.abs((V_values - mean_V) / std_V)
        critical_value = stats.norm.ppf((1 + confidence_level) / 2)
        poison_indicators = z_scores > critical_value
        poison_indicators = poison_indicators.astype(int)



        return (
            poison_indicators,
            {
                'confidence_interval': (ci_lower, ci_upper),
                'mean_lyapunov': mean_V,
                'std_lyapunov': std_V,
                'critical_value': critical_value,
                'z_scores': z_scores


            }
        )

    def plot_lyapunov_analysis(self, save_path: Optional[str] = None):
        """Plot comprehensive Lyapunov analysis with separated traces."""
        import matplotlib.pyplot as plt
        import seaborn as sns

        # Configure plotting style
        if self.visualization_style == 'seaborn':
            sns.set_style('whitegrid')
            sns.set_context('paper', font_scale=1.2)

        # Create figure with subplots
        fig = plt.figure(figsize=(20, 15))
        gs = GridSpec(3, 2, figure=fig)

        # Plot Lyapunov spectrum distribution
        ax1 = fig.add_subplot(gs[0, :])
        self._plot_trace_distributions(ax1)

        # Plot weight evolution
        ax2 = fig.add_subplot(gs[1, :])
        self._plot_weight_evolution(ax2)


        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

    def _plot_trace_distributions(self, ax):
        """Plot separate trace distributions with kernel density estimation."""
        trace_values = [trace['values'] for trace in self.trace_metadata]
        timestamps = [trace['timestamp'] for trace in self.trace_metadata]

        colors = sns.color_palette('husl', n_colors=len(trace_values))
        for i, values in enumerate(trace_values):
            kernel_density = stats.gaussian_kde(values)
            x_range = np.linspace(min(values), max(values), 100)
            density = kernel_density(x_range)
            label = f'Trace {i+1}\n({timestamps[i][:16]})'
            ax.plot(x_range, density, color=colors[i], linewidth=2,
                   alpha=0.7, label=label)
            ax.fill_between(x_range, density, alpha=0.2, color=colors[i])

        ax.set_xlabel('Lyapunov Values')
        ax.set_ylabel('Density')
        ax.set_title('Separate Trace Distributions')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True)

    def _plot_weight_evolution(self, ax):
        """Plot weight evolution analysis."""
        if self.weight_history:
            weight_norms = [np.linalg.norm(w) for w in self.weight_history]
            ax.plot(weight_norms, 'g-', linewidth=2)
            ax.set_xlabel('Time Steps')
            ax.set_ylabel('Weight Norm')
            ax.set_title('Weight Evolution Analysis')
            ax.grid(True)

    def _plot_stability_metrics(self, ax):
        """Plot stability metrics over time."""
        if self.stability_metrics:
            timestamps = [
                datetime.fromisoformat(m['timestamp'])
                for m in self.stability_metrics
            ]
            stability_values = [m['stability'] for m in self.stability_metrics]

            ax.plot(timestamps, stability_values, 'b-', linewidth=2)
            ax.set_xlabel('Time')
            ax.set_ylabel('Stability Metric')
            ax.set_title('System Stability Over Time')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True)



    def analyze_topological_transitivity(self) -> Dict[str, float]:
        """
        Analyze topological transitivity of the latent learning region.

        Returns:
            Dictionary containing transitivity metrics including:
            - mixing_index: Measure of state space mixing
            - recurrence_ratio: Ratio of recurrent states
            - ergodicity_score: Measure of ergodic behavior
        """
        if not self.trace_metadata:
            raise ValueError("No trace data available for analysis")

        # Extract Lyapunov values and compute derivatives
        lyapunov_values = np.concatenate([trace['values'] for trace in self.trace_metadata])
        d_lyapunov = np.diff(lyapunov_values)

        # Compute mixing index using correlation analysis
        mixing_index = self._compute_mixing_index(d_lyapunov)


        # Calculate recurrence ratio
        recurrence_ratio = self._calculate_recurrence_ratio(lyapunov_values)

        # Estimate ergodicity score
        ergodicity_score = self._estimate_ergodicity(lyapunov_values)

        return {
            'mixing_index': mixing_index,
            'recurrence_ratio': recurrence_ratio,
            'ergodicity_score': ergodicity_score
        }

    def _compute_mixing_index(self, derivatives: np.ndarray) -> float:
        """Compute mixing index using autocorrelation analysis."""
        auto_corr = np.correlate(derivatives, derivatives, mode='full')[len(derivatives)-1:]
        auto_corr = auto_corr / auto_corr[0]

        # Find first zero crossing
        zero_crossings = np.where(np.diff(np.signbit(auto_corr)))[0]
        if len(zero_crossings) == 0:
            return 0.0

        mixing_time = zero_crossings[0]
        return 1.0 / (1.0 + np.exp(-mixing_time))



    def _calculate_recurrence_ratio(self, values: np.ndarray) -> float:
        """Calculate recurrence ratio using Poincaré section analysis."""
        threshold = np.median(values)
        crossings = np.sum(np.diff((values > threshold).astype(int)))
        return crossings / len(values)

    def _estimate_ergodicity(self, values: np.ndarray) -> float:
        """Estimate ergodicity using spectral analysis."""
        fft_spectrum = np.abs(np.fft.fft(values))
        normalized_spectrum = fft_spectrum / np.sum(fft_spectrum)
        entropy = -np.sum(normalized_spectrum * np.log(normalized_spectrum + 1e-10))
        return entropy / np.log(len(values))


    def analyze_bifurcation(self, parameter_range: Tuple[float, float],
                       num_points: int = 1000,
                       iterations: int = 1000) -> Dict:
        """
        Perform advanced bifurcation analysis with stability metrics.

        Args:
          parameter_range: Tuple of (min_value, max_value) for parameter sweep
          num_points: Number of points to evaluate
          iterations: Number of iterations for each parameter value

        Returns:
          Dictionary containing bifurcation data and stability metrics
        """
        # Initialize arrays for bifurcation analysis
        params = np.linspace(parameter_range[0], parameter_range[1], num_points)
        bifurcation_data = []
        stability_metrics = []

        # Perform bifurcation analysis
        for param in params:
            # Initialize state
            x = np.random.random()

            # Iterate and collect data after transient
            values = []
            for i in range(iterations):
                # Update using logistic map for demonstration
                x = param * x * (1 - x)
                if i > iterations // 2:  # Skip transient
                   values.append(x)

            # Calculate stability metrics
            if values:
               stability = self._compute_stability(values)
               bifurcation_data.append({
                'parameter': param,
                'values': values,
                'stability': stability
                })
            stability_metrics.append(stability)

        return {
           'bifurcation_data': bifurcation_data,
           'stability_metrics': stability_metrics,
           'parameter_range': parameter_range
                  }

    def _compute_stability(self, values: List[float]) -> float:
        """
        Compute stability metric using Lyapunov exponent estimation.
        """
        if len(values) < 2:
           return 0.0

        # Compute local Lyapunov exponents
        lyapunovs = []
        for i in range(len(values)-1):
            # Approximate derivative
            dx = values[i+1] - values[i]
            # Local Lyapunov exponent
            lyapunov = np.log(abs(dx)) if dx != 0 else 0
            lyapunovs.append(lyapunov)

          # Average Lyapunov exponent
        return np.mean(lyapunovs)

    def plot_bifurcation_analysis(self, bifurcation_data: Dict,
                            save_path: Optional[str] = None):
        """
        Create comprehensive bifurcation analysis visualization with stability metrics.
        """
        import matplotlib.pyplot as plt
        import seaborn as sns

        # Configure plotting style
        sns.set_style('whitegrid')
        sns.set_context('paper', font_scale=1.2)

        # Create figure with subplots
        fig = plt.figure(figsize=(15, 10))
        gs = plt.GridSpec(2, 2, figure=fig)

        # Bifurcation diagram
        ax1 = fig.add_subplot(gs[0, :])
        self._plot_bifurcation_diagram(ax1, bifurcation_data)

        # Stability analysis
        ax2 = fig.add_subplot(gs[1, 0])
        self._plot_stability_metrics(ax2, bifurcation_data)

        # Phase space analysis
        ax3 = fig.add_subplot(gs[1, 1])
        self._plot_phase_space(ax3, bifurcation_data)

        plt.tight_layout()
        if save_path:
           plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

    def _plot_bifurcation_diagram(self, ax, bifurcation_data: Dict):
        """
        Plot the bifurcation diagram with color-coded stability regions.
        """
        data = bifurcation_data['bifurcation_data']
        params = [d['parameter'] for d in data]
        values = [d['values'] for d in data]
        stability = [d['stability'] for d in data]

        # Create colormap based on stability
        cmap = plt.cm.RdYlBu_r
        norm = plt.Normalize(min(stability), max(stability))

        # Plot points with stability-based coloring
        for i, param in enumerate(params):
           for value in values[i]:
               ax.scatter(param, value,
                      c=[cmap(norm(stability[i]))],
                      alpha=0.6, s=1)

        ax.set_xlabel('Parameter Value')
        ax.set_ylabel('State Value')
        ax.set_title('Bifurcation Diagram with Stability Metrics')

    def _plot_stability_metrics(self, ax, bifurcation_data: Dict):
        """
        Plot stability metrics across parameter space.
        """
        data = bifurcation_data['bifurcation_data']
        params = [d['parameter'] for d in data]
        stability = [d['stability'] for d in data]

        ax.plot(params, stability, 'b-', linewidth=2)
        ax.fill_between(params, stability, alpha=0.3)

        ax.set_xlabel('Parameter Value')
        ax.set_ylabel('Stability Metric')
        ax.set_title('System Stability Across Parameter Space')
        ax.grid(True)


    def _plot_phase_space(self, ax, bifurcation_data: Dict):
        """
        Plot phase space analysis showing system dynamics and attractors.

        Args:
          ax: Matplotlib axis object
          bifurcation_data: Dictionary containing bifurcation analysis results
        """
        data = bifurcation_data['bifurcation_data']

        # Extract values for phase space analysis
        params = [d['parameter'] for d in data]
        values = [d['values'] for d in data]
        stability = [d['stability'] for d in data]

        # Create phase space visualization
        for i, param in enumerate(params):
            if values[i]:  # Only plot if we have values
               # Plot trajectory in phase space
               ax.plot(values[i][:-1], values[i][1:],
                     'b.', alpha=0.1, markersize=1)

               # Highlight stable regions
               if stability[i] > 0:
                  ax.plot(values[i][:-1], values[i][1:],
                         'g.', alpha=0.2, markersize=1)

        ax.set_xlabel('Previous State')
        ax.set_ylabel('Current State')
        ax.set_title('Phase Space Analysis')
        ax.grid(True)

    def plot_detection_results(self, detection_stats: Dict, save_path: Optional[str] = None):
        """
         Plot detection results with color-coded array visualization.

         Args:
          detection_stats: Dictionary containing detection statistics
          save_path: Optional path to save the figure
        """
        import matplotlib.pyplot as plt
        import seaborn as sns
        import numpy as np
        from mpl_toolkits.axes_grid1 import make_axes_locatable

        # Configure plotting style
        sns.set_style('whitegrid')
        sns.set_context('paper', font_scale=1.2)

        # Create figure with constrained layout
        fig = plt.figure(figsize=(15, 10), layout="constrained")
        gs = plt.GridSpec(2, 2, figure=fig)

        # Create array of results
        results_array = np.array([
           [detection_stats['mean_lyapunov'],
           detection_stats['std_lyapunov'],
           detection_stats['critical_value']],
           [detection_stats['confidence_interval'][0],
           detection_stats['confidence_interval'][1],
           np.mean(detection_stats['z_scores'])]
                     ])

        # Plot results array with color coding
        ax1 = fig.add_subplot(gs[0, :])
        im = ax1.imshow(results_array, cmap='RdYlBu_r')

        # Add text annotations
        for i in range(results_array.shape[0]):
          for j in range(results_array.shape[1]):
              text_color = 'red' if i == 1 and j == 2 else 'black'
              ax1.text(j, i, f'{results_array[i, j]:.3f}',
                    ha='center', va='center', color=text_color)

        # Add labels
        ax1.set_xticks([0, 1, 2])
        ax1.set_yticks([0, 1])
        ax1.set_xticklabels(['Mean', 'Std', 'Critical Value'])
        ax1.set_yticklabels(['Metrics','Confidence'])
        ax1.set_title('Detection Statistics')

        # Create colorbar using axes_grid1
        divider = make_axes_locatable(ax1)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(im, cax=cax)

        # Plot z-scores distribution
        ax2 = fig.add_subplot(gs[1, 0])
        sns.histplot(detection_stats['z_scores'], ax=ax2, color='blue', alpha=0.6)
        ax2.axvline(detection_stats['critical_value'], color='red', linestyle='--',
                label='Critical Value')
        ax2.set_title('Z-scores Distribution')
        ax2.legend()

        # Plot confidence intervals
        ax3 = fig.add_subplot(gs[1, 1])
        ax3.plot([1, 2], detection_stats['confidence_interval'], 'b-', linewidth=2)
        ax3.fill_between([1, 2],
                     [detection_stats['confidence_interval'][0]]*2,
                    [detection_stats['confidence_interval'][1]]*2,
                     alpha=0.3)
        ax3.set_xticks([1, 2])
        ax3.set_xticklabels(['Lower', 'Upper'])
        ax3.set_title('Confidence Intervals')

        # Adjust layout
        fig.get_layout_engine().set(w_pad=4/72, h_pad=4/72, hspace=0.2, wspace=0.2)

        if save_path:
          plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()


    def kolmogorov_equation(self, x: np.ndarray, t: float) -> np.ndarray:
        """
        Implements the Kolmogorov equation for stability analysis.

        The Kolmogorov equation is used to track the probability density
        distribution of the system over time. It combines the drift and
        diffusion terms to model the system's evolution.

        Args:
          x: State vector (numpy array)
          t: Time point for evaluation

        Returns:
         numpy array containing the time derivative of the probability density
        """
        # Get drift and diffusion terms from the Lyapunov function
        drift = self._compute_drift(x)
        diffusion = self._compute_diffusion(x)

        # Compute spatial derivatives
        dx_drift = self._compute_spatial_derivative(drift)
        dx_diffusion = self._compute_spatial_derivative(diffusion)

        # Compute the Kolmogorov equation
        # ∂P/∂t = -∇ · (μP) + (1/2)∇ · (∇ · (ΣP))
        # where μ is drift, Σ is diffusion matrix
        kolmogorov_term = (
        -np.sum(dx_drift) +
        0.5 * np.sum(dx_diffusion)
           )

        return kolmogorov_term

    def _compute_drift(self, x: np.ndarray) -> np.ndarray:
        """
        Compute the drift term (μ) of the Kolmogorov equation.

        The drift term represents the deterministic component of the system's dynamics.
        """
        # Use the Lyapunov function's gradient as drift
        drift = np.gradient(self.lyapunov_function(x))
        return drift

    def _compute_diffusion(self, x: np.ndarray) -> np.ndarray:
        """
        Compute the diffusion term (Σ) of the Kolmogorov equation.

        The diffusion term represents the stochastic component of the system's dynamics.
        """
        # Estimate diffusion from the system's stability metrics
        stability = self.temporal_stability_analysis()
        diffusion = stability * np.eye(x.shape[-1])
        return diffusion

    def _compute_spatial_derivative(self, field: np.ndarray) -> np.ndarray:
        """
        Compute the spatial derivative of a field using finite differences.

         Args:
         field: Input field to compute derivative of

        Returns:
         Spatial derivative of the input field
        """
        # Use central differences for better accuracy
        dx = np.gradient(field)
        return dx

    def compute_lyapunov_spectrum(self, x: np.ndarray) -> List[float]:
        """
        Compute Lyapunov spectrum using tangent space dynamics.

        Args:
            x: Input data array

        Returns:
            List of Lyapunov exponents
        """
        processed_x = self.preprocess_data(x)

        # Compute Jacobian matrix
        jacobian = self._compute_jacobian(processed_x)

        # Calculate Lyapunov exponents using QR decomposition
        spectrum = []
        for i in range(self.temporal_window):
            if i < len(jacobian):
                q, r = np.linalg.qr(jacobian[i])
                spectrum.append(np.log(np.abs(np.diag(r))))

        # Record spectrum history
        self.spectrum_history.append(spectrum)

        return spectrum

    def _compute_jacobian(self, x: np.ndarray) -> np.ndarray:
        """Compute Jacobian matrix of the system."""
        eps = 1e-6
        n = len(x)
        jacobian = np.zeros((n, n))

        for i in range(n):
            dx = np.zeros_like(x)
            dx[i] += eps
            f_plus = self.lyapunov_function(x + dx)
            f_minus = self.lyapunov_function(x - dx)
            jacobian[:, i] = (f_plus - f_minus) / (2 * eps)

        return jacobian

    def visualize_stability_analysis(self):
        """Generate comprehensive stability analysis plots."""
        fig = plt.figure(figsize=(15, 10))

        # Plot Lyapunov values over time
        ax1 = fig.add_subplot(221)
        ax1.plot(self.lyapunov_values)
        ax1.set_title('Lyapunov Values Over Time')
        ax1.set_xlabel('Time Step')
        ax1.set_ylabel('V(x)')

        # Plot Lyapunov spectrum
        ax2 = fig.add_subplot(222)
        if self.spectrum_history:
            spectrum_mean = np.mean(self.spectrum_history, axis=0)
            ax2.bar(range(len(spectrum_mean)), spectrum_mean)
            ax2.set_title('Lyapunov Spectrum')
            ax2.set_xlabel('Index')
            ax2.set_ylabel('Exponent Value')

        # Plot weight evolution
        ax3 = fig.add_subplot(223)
        if self.weight_history:
            weight_norms = [np.linalg.norm(w) for w in self.weight_history]
            ax3.plot(weight_norms)
            ax3.set_title('Weight Evolution')
            ax3.set_xlabel('Time Step')
            ax3.set_ylabel('Weight Norm')

        # Plot stability metrics
        ax4 = fig.add_subplot(224)
        if self.stability_metrics:
            stability_values = [m['stability'] for m in self.stability_metrics]
            ax4.plot(stability_values)
            ax4.set_title('Stability Metrics')
            ax4.set_xlabel('Time Step')
            ax4.set_ylabel('Stability Score')

        plt.tight_layout()
        return fig



In [ ]:
# Initialize detector
detector = EnhancedMetaLearningLyapunov(
    input_shape=(1, 124, 129),
    learning_rate=0.0001,
    regularization=0.01,
    temporal_window=5,
    spatial_pooling='avg',
    trace_separation='time',
    robustness_level='high',
    detection_threshold=0.1,
    visualization_style='seaborn',
    poisoning_detection_method='statistical',
    anomaly_sensitivity= 0.05
    )


# Adapt to task
loss, diagnostics = detector.adapt_to_task(x_train_mix, y_train_mix, num_steps=20)


# Detect poisoning
poison_indicators, detection_stats = detector.detect_poisoning(
    x_test=x_test_mix ,
    threshold=0.1,
    confidence_level=0.99

)

# Visualize results
detector.plot_lyapunov_analysis('lyapunov_analysis.png')
detector.plot_detection_results(detection_stats, 'detection_results.png')

# Analyze topological transitivity
transitivity_metrics = detector.analyze_topological_transitivity()
print("Transitivity Metrics:", transitivity_metrics)



# Perform bifurcation analysis
bifurcation_results = detector.analyze_bifurcation(
    parameter_range=(0.7, 4.0),  # Range for bifurcation parameter
    num_points=1500,            # Resolution of analysis
    iterations=20             # Number of iterations per point
)

# Visualize results
detector.plot_bifurcation_analysis(bifurcation_results,
                                 save_path='bifurcation_analysis.png')


# Perform enhanced detection
poison_indicators, detection_results = detector.detect_poisoning(
    x_test=x_test_mix,
    threshold=0.1,
    confidence_level=0.99
)

print("detection_results:", detection_results)

# Visualize results
detector.plot_detection_results(detection_results)
